In [ ]:
# Install required packages (run once)
!pip install pyspark==3.5.0 -q

## Part A: Data Ingestion and Storage Design

### A.1 SparkSession Configuration
Configured for optimal performance with:
- **Memory settings**: Increased driver/executor memory for large datasets
- **Parallelism**: Tuned for local machine with shuffle partitions
- **Serialization**: Kryo serializer for better performance
- **Adaptive Query Execution**: Enabled for dynamic optimization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark import StorageLevel
import time

# =============================================================================
# SPARKSESSION CONFIGURATION - Optimized for Data Engineering Pipeline
# =============================================================================

spark = SparkSession.builder \
    .appName("311-Data-Engineering-Pipeline") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.broadcastTimeout", "600") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .getOrCreate()

# Set log level to reduce verbosity
spark.sparkContext.setLogLevel("WARN")

# Verify configuration
print("=" * 60)
print("SPARK CONFIGURATION SUMMARY")
print("=" * 60)
print(f"App Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")
print(f"Spark Version: {spark.version}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"AQE Enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print("=" * 60)

### A.2 Data Ingestion with Validation
Loading 311 Service Request data from Huggingface with:
- Sample size of 200,000 records
- Schema validation during ingestion
- Null/missing value detection
- Data quality metrics reporting

In [ ]:
from datasets import load_dataset
import pandas as pd

# =============================================================================
# DATA INGESTION - Load 311 Data from HuggingFace
# =============================================================================

print("Loading 311 Service Request data from HuggingFace...")
start_time = time.time()

# Load dataset from HuggingFace
SAMPLE_SIZE = 200000
dataset = load_dataset("edwinjue/311-data-2020", split=f"train[:{SAMPLE_SIZE}]")

# Convert to Pandas first for data inspection
pandas_df = dataset.to_pandas()
print(f"Data loaded in {time.time() - start_time:.2f} seconds")
print(f"Records loaded: {len(pandas_df):,}")
print(f"Columns: {len(pandas_df.columns)}")

# Display sample
pandas_df.head()

In [ ]:
# =============================================================================
# SCHEMA DEFINITION - Define explicit schema for data validation
# =============================================================================

# Define explicit schema based on 311 data structure
schema = StructType([
    StructField("unique_key", StringType(), True),
    StructField("created_date", StringType(), True),
    StructField("closed_date", StringType(), True),
    StructField("agency", StringType(), True),
    StructField("agency_name", StringType(), True),
    StructField("complaint_type", StringType(), True),
    StructField("descriptor", StringType(), True),
    StructField("location_type", StringType(), True),
    StructField("incident_zip", StringType(), True),
    StructField("incident_address", StringType(), True),
    StructField("street_name", StringType(), True),
    StructField("cross_street_1", StringType(), True),
    StructField("cross_street_2", StringType(), True),
    StructField("intersection_street_1", StringType(), True),
    StructField("intersection_street_2", StringType(), True),
    StructField("address_type", StringType(), True),
    StructField("city", StringType(), True),
    StructField("landmark", StringType(), True),
    StructField("facility_type", StringType(), True),
    StructField("status", StringType(), True),
    StructField("due_date", StringType(), True),
    StructField("resolution_description", StringType(), True),
    StructField("resolution_action_updated_date", StringType(), True),
    StructField("community_board", StringType(), True),
    StructField("bbl", StringType(), True),
    StructField("borough", StringType(), True),
    StructField("x_coordinate_state_plane", DoubleType(), True),
    StructField("y_coordinate_state_plane", DoubleType(), True),
    StructField("open_data_channel_type", StringType(), True),
    StructField("park_facility_name", StringType(), True),
    StructField("park_borough", StringType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("taxi_company_borough", StringType(), True),
    StructField("taxi_pick_up_location", StringType(), True),
    StructField("bridge_highway_name", StringType(), True),
    StructField("bridge_highway_direction", StringType(), True),
    StructField("road_ramp", StringType(), True),
    StructField("bridge_highway_segment", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("location", StringType(), True),
])

print(f"Schema defined with {len(schema.fields)} fields")

In [ ]:
# =============================================================================
# CONVERT TO SPARK DATAFRAME with Data Validation
# =============================================================================

print("Converting to Spark DataFrame with data validation...")
start_time = time.time()

# Create Spark DataFrame from Pandas
# Using Arrow for efficient conversion (configured in SparkSession)
df_raw = spark.createDataFrame(pandas_df)

# Repartition for parallel processing
df_raw = df_raw.repartition(8)

conversion_time = time.time() - start_time
print(f"Conversion completed in {conversion_time:.2f} seconds")

# Display schema
print("\n" + "=" * 60)
print("INFERRED SCHEMA")
print("=" * 60)
df_raw.printSchema()

In [ ]:
# =============================================================================
# DATA VALIDATION - Check data quality at ingestion
# =============================================================================

def validate_dataframe(df, df_name="DataFrame"):
    """Comprehensive data validation function"""
    print(f"\n{'=' * 60}")
    print(f"DATA VALIDATION REPORT: {df_name}")
    print(f"{'=' * 60}")

    total_rows = df.count()
    total_cols = len(df.columns)

    print(f"\n1. BASIC STATISTICS:")
    print(f"   - Total Records: {total_rows:,}")
    print(f"   - Total Columns: {total_cols}")

    # Check for duplicate keys
    unique_keys = df.select("SRNumber").distinct().count()
    duplicate_count = total_rows - unique_keys
    print(f"\n2. DUPLICATE CHECK:")
    print(f"   - Unique Keys: {unique_keys:,}")
    print(f"   - Duplicate Records: {duplicate_count:,}")

    # Null value analysis
    print(f"\n3. NULL VALUE ANALYSIS (Top 10 columns with nulls):")
    null_counts = []
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            null_pct = (null_count / total_rows) * 100
            null_counts.append((col_name, null_count, null_pct))

    null_counts.sort(key=lambda x: x[1], reverse=True)
    for col_name, count, pct in null_counts[:10]:
        print(f"   - {col_name}: {count:,} ({pct:.2f}%)")

    # Data type validation
    print(f"\n4. DATA TYPE SUMMARY:")
    type_summary = {}
    for field in df.schema.fields:
        dtype = str(field.dataType)
        type_summary[dtype] = type_summary.get(dtype, 0) + 1
    for dtype, count in type_summary.items():
        print(f"   - {dtype}: {count} columns")

    return {
        "total_rows": total_rows,
        "total_cols": total_cols,
        "duplicates": duplicate_count,
        "null_summary": null_counts
    }

# Run validation
validation_results = validate_dataframe(df_raw, "311 Service Requests Raw Data")

### A.3 Partitioning Strategy Aligned with Query Patterns
**Partitioning Justification:**
- **Borough-based partitioning**: Most queries filter by geographic region (borough)
- **Agency-based partitioning**: Common queries filter by responding agency
- Reduces data scanning during queries (partition pruning)
- Enables efficient parallel processing per partition

**Trade-offs:**
- Too many partitions = small files (overhead)
- Too few partitions = large files (less parallelism)
- Chosen columns have moderate cardinality (5 boroughs, ~30 agencies)

In [ ]:
# =============================================================================
# PARTITIONING STRATEGY ANALYSIS
# =============================================================================

print("Analyzing partitioning candidates...")

# Analyze cardinality of potential partition columns
partition_candidates = ["borough", "agency", "complaint_type", "status"]

print("\n" + "=" * 60)
print("PARTITION COLUMN CARDINALITY ANALYSIS")
print("=" * 60)

for col_name in partition_candidates:
    if col_name in df_raw.columns:
        distinct_count = df_raw.select(col_name).distinct().count()
        null_count = df_raw.filter(F.col(col_name).isNull()).count()
        print(f"\n{col_name}:")
        print(f"   - Distinct Values: {distinct_count}")
        print(f"   - Null Values: {null_count:,}")

        # Show top 5 values
        top_values = df_raw.groupBy(col_name) \
            .count() \
            .orderBy(F.desc("count")) \
            .limit(5) \
            .collect()
        print(f"   - Top 5 Values:")
        for row in top_values:
            print(f"      {row[col_name]}: {row['count']:,}")

In [ ]:
# =============================================================================
# DATA CLEANING AND TRANSFORMATION - FIXED COLUMN MAPPING
# =============================================================================

print("Performing data cleaning and transformation...")
start_time = time.time()

# Map columns based on the actual inferred schema
# Note: This specific dataset uses 'Owner' as the agency and doesn't have a 'borough' column
# We will use 'ZipCode' or 'Address' as proxies or set to 'UNKNOWN' if missing.

df_clean = df_raw \
    .withColumn("borough_clean", F.lit("UNKNOWN")) \
    .withColumn("agency_clean",
                F.when(F.col("Owner").isNull(), "UNKNOWN")
                .otherwise(F.upper(F.trim(F.col("Owner"))))) \
    .withColumn("created_date_parsed",
                F.to_timestamp(F.col("CreatedDate"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("closed_date_parsed",
                F.to_timestamp(F.col("ClosedDate"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("created_year", F.year("created_date_parsed")) \
    .withColumn("created_month", F.month("created_date_parsed")) \
    .withColumn("created_day", F.dayofmonth("created_date_parsed")) \
    .withColumn("created_hour", F.hour("created_date_parsed")) \
    .withColumn("created_day_of_week", F.dayofweek("created_date_parsed")) \
    .withColumn("resolution_hours",
                F.when(F.col("closed_date_parsed").isNotNull(),
                       (F.unix_timestamp("closed_date_parsed") -
                        F.unix_timestamp("created_date_parsed")) / 3600)
                .otherwise(None))

# Select relevant columns for analysis using the correct names
df_processed = df_clean.select(
    F.col("SRNumber").alias("unique_key"),
    "created_date_parsed",
    "closed_date_parsed",
    "agency_clean",
    F.col("Owner").alias("agency_name"),
    F.col("RequestType").alias("complaint_type"),
    "borough_clean",
    "Status",
    "resolution_hours",
    "created_year",
    "created_month",
    "created_day",
    "created_hour",
    "created_day_of_week",
    "Latitude",
    "Longitude"
)

print(f"Data transformation completed in {time.time() - start_time:.2f} seconds")
print(f"Processed columns: {len(df_processed.columns)}")
df_processed.printSchema()

### A.4 Storage Format: Parquet with Justification

**Why Parquet?**
1. **Columnar Storage**: Efficient for analytical queries reading specific columns
2. **Compression**: Snappy compression reduces storage by 60-80%
3. **Schema Evolution**: Supports adding new columns without breaking existing queries
4. **Predicate Pushdown**: Filter operations pushed to storage layer
5. **Spark Native**: Native integration with Spark, optimized for big data
6. **Metadata**: Self-describing format with embedded schema

**Comparison with alternatives:**
| Format | Read Speed | Write Speed | Compression | Schema | Use Case |
|--------|------------|-------------|-------------|--------|----------|
| **Parquet** | Fast | Moderate | Excellent | Embedded | Analytics |
| CSV | Slow | Fast | Poor | External | Interchange |
| JSON | Slow | Fast | Poor | Flexible | Semi-structured |
| ORC | Fast | Moderate | Excellent | Embedded | Hive workloads |

In [ ]:
import os
import shutil

# =============================================================================
# SAVE TO PARQUET WITH PARTITIONING
# =============================================================================

# Define output paths
OUTPUT_DIR = "311_data_parquet"
PARTITIONED_DIR = f"{OUTPUT_DIR}/partitioned_by_borough"
NON_PARTITIONED_DIR = f"{OUTPUT_DIR}/non_partitioned"

# Clean up existing directories
for dir_path in [PARTITIONED_DIR, NON_PARTITIONED_DIR]:
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)

print("Writing Parquet files...")
print("=" * 60)

# Write non-partitioned Parquet for comparison
print("\n1. Writing NON-PARTITIONED Parquet...")
start_time = time.time()
df_processed.write \
    .mode("overwrite") \
    .parquet(NON_PARTITIONED_DIR)
non_part_time = time.time() - start_time
print(f"   Completed in {non_part_time:.2f} seconds")

# Write partitioned Parquet (by borough for query optimization)
print("\n2. Writing PARTITIONED Parquet (by borough_clean)...")
start_time = time.time()
df_processed.write \
    .mode("overwrite") \
    .partitionBy("borough_clean") \
    .parquet(PARTITIONED_DIR)
part_time = time.time() - start_time
print(f"   Completed in {part_time:.2f} seconds")

# Compare file sizes
def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

non_part_size = get_dir_size(NON_PARTITIONED_DIR) / (1024 * 1024)
part_size = get_dir_size(PARTITIONED_DIR) / (1024 * 1024)

print(f"\n" + "=" * 60)
print("PARQUET STORAGE SUMMARY")
print("=" * 60)
print(f"Non-Partitioned Size: {non_part_size:.2f} MB")
print(f"Partitioned Size: {part_size:.2f} MB")
print(f"Compression Codec: Snappy (default)")
print(f"Partitions Created: {len(os.listdir(PARTITIONED_DIR))}")

---
## Part B: Distributed Data Processing Pipeline

### B.1 Broadcast Join Implementation
**When to use Broadcast Joins:**
- When joining a large DataFrame with a small DataFrame (< 10MB default)
- Small table is replicated to all executor nodes
- Eliminates expensive shuffle operations
- Use `broadcast()` hint to force broadcast even if auto-broadcast is disabled

**Use Case:** Joining with lookup tables (agency metadata, borough info)

In [ ]:
# =============================================================================
# BROADCAST JOIN IMPLEMENTATION
# =============================================================================

from pyspark.sql.functions import broadcast

# Read from Parquet (demonstrating partition pruning)
print("Reading from partitioned Parquet...")
df_main = spark.read.parquet(PARTITIONED_DIR)
print(f"Main DataFrame records: {df_main.count():,}")

# Create a small lookup table (borough metadata) - SMALL TABLE
borough_data = [
    ("MANHATTAN", "New York County", 1_628_706, 22.83),
    ("BROOKLYN", "Kings County", 2_559_903, 69.5),
    ("QUEENS", "Queens County", 2_253_858, 108.53),
    ("BRONX", "Bronx County", 1_418_207, 42.1),
    ("STATEN ISLAND", "Richmond County", 476_143, 58.37),
    ("UNKNOWN", "Unknown", 0, 0.0)
]

borough_schema = StructType([
    StructField("borough_name", StringType(), False),
    StructField("county_name", StringType(), False),
    StructField("population", IntegerType(), False),
    StructField("area_sq_miles", DoubleType(), False)
])

df_borough_lookup = spark.createDataFrame(borough_data, schema=borough_schema)
print(f"\nBorough lookup table records: {df_borough_lookup.count()}")
df_borough_lookup.show()

In [ ]:
# =============================================================================
# BROADCAST JOIN vs REGULAR JOIN COMPARISON
# =============================================================================

print("=" * 60)
print("BROADCAST JOIN vs REGULAR JOIN COMPARISON")
print("=" * 60)

# Regular Join (may cause shuffle)
print("\n1. REGULAR JOIN (with potential shuffle)...")
start_time = time.time()
df_regular_join = df_main.join(
    df_borough_lookup,
    df_main["borough_clean"] == df_borough_lookup["borough_name"],
    "left"
)
# Force evaluation
regular_count = df_regular_join.count()
regular_time = time.time() - start_time
print(f"   Records: {regular_count:,}")
print(f"   Time: {regular_time:.2f} seconds")

# Broadcast Join (no shuffle - small table broadcast to all nodes)
print("\n2. BROADCAST JOIN (no shuffle)...")
start_time = time.time()
df_broadcast_join = df_main.join(
    broadcast(df_borough_lookup),  # Hint to broadcast small table
    df_main["borough_clean"] == df_borough_lookup["borough_name"],
    "left"
)
# Force evaluation
broadcast_count = df_broadcast_join.count()
broadcast_time = time.time() - start_time
print(f"   Records: {broadcast_count:,}")
print(f"   Time: {broadcast_time:.2f} seconds")

# Show speed improvement
if regular_time > 0:
    improvement = ((regular_time - broadcast_time) / regular_time) * 100
    print(f"\n   Performance Improvement: {improvement:.1f}%")

# Explain plans comparison
print("\n" + "=" * 60)
print("EXECUTION PLAN - BROADCAST JOIN")
print("=" * 60)
df_broadcast_join.explain(mode="simple")

### B.2 Memory Management: persist() / unpersist() Strategy

**Caching Best Practices:**
- `cache()` = `persist(StorageLevel.MEMORY_ONLY)` - Fastest but may spill
- `persist(MEMORY_AND_DISK)` - Spills to disk if memory full
- `persist(MEMORY_ONLY_SER)` - Serialized, less memory, more CPU
- Always `unpersist()` when done to free memory

**When to Cache:**
1. DataFrame used multiple times in same session
2. After expensive transformations
3. Before iterative algorithms

**When NOT to Cache:**
1. Single-use DataFrames
2. Very small DataFrames (broadcast instead)
3. When memory is constrained

In [ ]:
# =============================================================================
# MEMORY MANAGEMENT DEMONSTRATION
# =============================================================================

print("=" * 60)
print("MEMORY MANAGEMENT: PERSIST/UNPERSIST DEMONSTRATION")
print("=" * 60)

# Use the enriched DataFrame
df_enriched = df_broadcast_join.select(
    "unique_key", "borough_clean", "agency_clean", "complaint_type",
    "created_date_parsed", "resolution_hours", "status",
    "county_name", "population", "area_sq_miles"
)

# Demonstrate different storage levels
print("\n1. STORAGE LEVEL: MEMORY_AND_DISK (Recommended for reuse)")
df_enriched.persist(StorageLevel.MEMORY_AND_DISK)

# Force materialization
start_time = time.time()
count1 = df_enriched.count()
first_time = time.time() - start_time
print(f"   First count (with computation): {count1:,} in {first_time:.2f}s")

# Second access (from cache)
start_time = time.time()
count2 = df_enriched.count()
cached_time = time.time() - start_time
print(f"   Second count (from cache): {count2:,} in {cached_time:.2f}s")

if first_time > 0:
    speedup = first_time / cached_time if cached_time > 0 else float('inf')
    print(f"   Cache Speedup: {speedup:.1f}x faster")

# Show storage info
print("\n2. CACHED DATAFRAMES IN MEMORY:")
print(f"   Is Cached: {df_enriched.is_cached}")
print(f"   Storage Level: {df_enriched.storageLevel}")

In [ ]:
# =============================================================================
# REUSE CACHED DATA FOR MULTIPLE AGGREGATIONS
# =============================================================================

print("\n3. MULTIPLE AGGREGATIONS ON CACHED DATA:")

# Aggregation 1: Complaints by borough
print("\n   Aggregation 1 - Complaints by Borough:")
start_time = time.time()
borough_summary = df_enriched.groupBy("borough_clean", "county_name", "population") \
    .agg(
        F.count("*").alias("complaint_count"),
        F.avg("resolution_hours").alias("avg_resolution_hours")
    ) \
    .withColumn("complaints_per_capita",
                F.col("complaint_count") / F.col("population") * 1000) \
    .orderBy(F.desc("complaint_count"))
borough_summary.show()
agg1_time = time.time() - start_time
print(f"   Time: {agg1_time:.2f}s")

# Aggregation 2: Top complaint types
print("\n   Aggregation 2 - Top Complaint Types:")
start_time = time.time()
complaint_summary = df_enriched.groupBy("complaint_type") \
    .agg(
        F.count("*").alias("count"),
        F.avg("resolution_hours").alias("avg_resolution_hrs")
    ) \
    .orderBy(F.desc("count")) \
    .limit(10)
complaint_summary.show()
agg2_time = time.time() - start_time
print(f"   Time: {agg2_time:.2f}s")

# Aggregation 3: Agency performance
print("\n   Aggregation 3 - Agency Performance:")
start_time = time.time()
agency_summary = df_enriched.groupBy("agency_clean") \
    .agg(
        F.count("*").alias("total_requests"),
        F.avg("resolution_hours").alias("avg_resolution_hrs"),
        F.sum(F.when(F.col("status") == "Closed", 1).otherwise(0)).alias("closed_count")
    ) \
    .withColumn("closure_rate",
                F.col("closed_count") / F.col("total_requests") * 100) \
    .orderBy(F.desc("total_requests")) \
    .limit(10)
agency_summary.show()
agg3_time = time.time() - start_time
print(f"   Time: {agg3_time:.2f}s")

In [ ]:
# =============================================================================
# PROPER MEMORY CLEANUP - UNPERSIST
# =============================================================================

print("\n4. MEMORY CLEANUP:")
print(f"   Before unpersist - Is Cached: {df_enriched.is_cached}")

# Unpersist to free memory
df_enriched.unpersist()

print(f"   After unpersist - Is Cached: {df_enriched.is_cached}")
print("   Memory released successfully!")

# Force garbage collection
spark.catalog.clearCache()
print("   All cached tables cleared.")

### B.3 Error Handling and Data Lineage

**Error Handling Strategies:**
1. Try-except blocks around Spark operations
2. Data quality checks with assertions
3. Graceful degradation for partial failures
4. Logging for debugging and monitoring

**Data Lineage:**
- Track transformations from source to destination
- Document data flow for auditability
- Use explain() to understand execution plans

In [18]:
# =============================================================================
# ERROR HANDLING AND DATA LINEAGE
# =============================================================================

import logging
from datetime import datetime

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("DataPipeline")

class DataPipeline:
    """Data pipeline with error handling and lineage tracking"""

    def __init__(self, name):
        self.name = name
        self.lineage = []
        self.start_time = datetime.now()
        self.errors = []

    def log_step(self, step_name, input_count=None, output_count=None, details=None):
        """Log transformation step for lineage tracking"""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "step": step_name,
            "input_count": input_count,
            "output_count": output_count,
            "details": details
        }
        self.lineage.append(entry)
        logger.info(f"Step: {step_name} | Input: {input_count} | Output: {output_count}")

    def safe_transform(self, df, transform_func, step_name):
        """Safely apply transformation with error handling"""
        try:
            input_count = df.count()
            result_df = transform_func(df)
            output_count = result_df.count()
            self.log_step(step_name, input_count, output_count)
            return result_df
        except Exception as e:
            error_msg = f"Error in {step_name}: {str(e)}"
            self.errors.append(error_msg)
            logger.error(error_msg)
            return df  # Return original on error

    def get_lineage_report(self):
        """Generate data lineage report"""
        print("\n" + "=" * 60)
        print(f"DATA LINEAGE REPORT: {self.name}")
        print("=" * 60)
        print(f"Pipeline Start: {self.start_time}")
        print(f"Total Steps: {len(self.lineage)}")
        print(f"Errors: {len(self.errors)}")
        print("\nTransformation Flow:")
        print("-" * 40)
        for i, step in enumerate(self.lineage, 1):
            print(f"{i}. {step['step']}")
            print(f"   Input: {step['input_count']:,} -> Output: {step['output_count']:,}")
        return self.lineage

# Create pipeline instance
pipeline = DataPipeline("311_Service_Requests_ETL")

In [ ]:
# =============================================================================
# PIPELINE EXECUTION WITH ERROR HANDLING
# =============================================================================

# Reload data for pipeline demonstration
df_source = spark.read.parquet(PARTITIONED_DIR)

# Define transformation functions
def filter_valid_records(df):
    """Filter records with valid borough and complaint type"""
    return df.filter(
        (F.col("borough_clean").isNotNull()) &
        (F.col("borough_clean") != "UNKNOWN") &
        (F.col("complaint_type").isNotNull())
    )

def add_derived_columns(df):
    """Add derived analytical columns"""
    return df.withColumn(
        "is_resolved",
        F.when(F.col("status") == "Closed", True).otherwise(False)
    ).withColumn(
        "resolution_category",
        F.when(F.col("resolution_hours") < 24, "Fast (<24hrs)")
         .when(F.col("resolution_hours") < 72, "Medium (24-72hrs)")
         .when(F.col("resolution_hours") < 168, "Slow (72hrs-1week)")
         .otherwise("Very Slow (>1week)")
    )

def aggregate_by_borough(df):
    """Aggregate metrics by borough"""
    return df.groupBy("borough_clean").agg(
        F.count("*").alias("total_complaints"),
        F.sum(F.when(F.col("is_resolved"), 1).otherwise(0)).alias("resolved_count"),
        F.avg("resolution_hours").alias("avg_resolution_hours")
    ).withColumn(
        "resolution_rate",
        F.round(F.col("resolved_count") / F.col("total_complaints") * 100, 2)
    )

# Execute pipeline with tracking
print("Executing data pipeline with error handling...")
print("=" * 60)

df_step1 = pipeline.safe_transform(df_source, filter_valid_records, "Filter Valid Records")
df_step2 = pipeline.safe_transform(df_step1, add_derived_columns, "Add Derived Columns")
df_step3 = pipeline.safe_transform(df_step2, aggregate_by_borough, "Aggregate by Borough")

# Show results
print("\nPipeline Output:")
df_step3.show()

# Generate lineage report
pipeline.get_lineage_report()

---
## Part C: Performance Optimization

### C.1 DataFrame vs RDD Usage Justification

| Aspect | DataFrame | RDD |
|--------|-----------|-----|
| **Optimization** | Catalyst optimizer, Tungsten execution | No automatic optimization |
| **Schema** | Structured with types | Schema-less |
| **Performance** | 10-100x faster due to optimization | Slower, more overhead |
| **API** | SQL-like, easier to use | Functional, more flexible |
| **Use Case** | Structured data, SQL queries | Unstructured, complex logic |

**Recommendation**: Use DataFrame API whenever possible for better performance and readability. Use RDD only for:
- Custom partitioning logic
- Low-level transformations
- Working with unstructured data

In [ ]:
# =============================================================================
# DATAFRAME vs RDD PERFORMANCE COMPARISON
# =============================================================================

print("=" * 60)
print("DATAFRAME vs RDD PERFORMANCE COMPARISON")
print("=" * 60)

# Reload data
df_perf = spark.read.parquet(PARTITIONED_DIR)

# Task: Count complaints by borough and get average resolution time

# Method 1: DataFrame API (Optimized)
print("\n1. DATAFRAME API:")
start_time = time.time()
df_result = df_perf.groupBy("borough_clean") \
    .agg(
        F.count("*").alias("count"),
        F.avg("resolution_hours").alias("avg_hours")
    ) \
    .orderBy(F.desc("count"))
df_count = df_result.count()  # Force evaluation
df_result.show()
df_time = time.time() - start_time
print(f"Time: {df_time:.3f} seconds")

# Method 2: RDD API (Less optimized)
print("\n2. RDD API:")
start_time = time.time()
rdd_result = df_perf.rdd \
    .map(lambda row: (row.borough_clean, (1, row.resolution_hours or 0))) \
    .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])) \
    .map(lambda x: (x[0], x[1][0], x[1][1] / x[1][0] if x[1][0] > 0 else 0)) \
    .collect()
rdd_time = time.time() - start_time

print(f"Borough | Count | Avg Hours")
print("-" * 40)
for item in sorted(rdd_result, key=lambda x: -x[1])[:10]:
    print(f"{item[0]:15} | {item[1]:6} | {item[2]:.2f}")
print(f"\nTime: {rdd_time:.3f} seconds")

# Comparison
print(f"\n" + "=" * 60)
print("PERFORMANCE COMPARISON")
print("=" * 60)
print(f"DataFrame Time: {df_time:.3f}s")
print(f"RDD Time: {rdd_time:.3f}s")
speedup = rdd_time / df_time if df_time > 0 else 0
print(f"DataFrame is {speedup:.1f}x faster than RDD!")
print("\nRECOMMENDATION: Use DataFrame API for structured data operations")

### C.2 Caching Strategy Documentation

**Storage Levels Available:**
| Level | Space | CPU | In Memory | On Disk | Serialized |
|-------|-------|-----|-----------|---------|------------|
| MEMORY_ONLY | High | Low | Yes | No | No |
| MEMORY_AND_DISK | High | Medium | Yes | Yes | No |
| MEMORY_ONLY_SER | Low | High | Yes | No | Yes |
| MEMORY_AND_DISK_SER | Low | High | Yes | Yes | Yes |
| DISK_ONLY | Low | High | No | Yes | Yes |

**Our Strategy:**
- Use `MEMORY_AND_DISK` for frequently accessed DataFrames
- Unpersist immediately after use to free memory
- Monitor memory via Spark UI Storage tab

In [ ]:
# =============================================================================
# CACHING STRATEGY DEMONSTRATION
# =============================================================================

print("=" * 60)
print("CACHING STRATEGY COMPARISON")
print("=" * 60)

# Prepare test DataFrame
df_cache_test = df_perf.select(
    "unique_key", "borough_clean", "complaint_type",
    "resolution_hours", "status", "agency_clean"
).filter(F.col("resolution_hours").isNotNull())

# Test different storage levels using valid Spark 3.5.0 constants
storage_levels = [
    ("MEMORY_ONLY", StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK),
    ("DISK_ONLY", StorageLevel.DISK_ONLY)
]

results = []

for level_name, storage_level in storage_levels:
    print(f"\nTesting: {level_name}")

    # Clear any existing cache
    spark.catalog.clearCache()

    # Persist with this level
    df_test = df_cache_test.persist(storage_level)

    # First access (cold)
    start_time = time.time()
    count1 = df_test.count()
    cold_time = time.time() - start_time

    # Second access (warm/cached)
    start_time = time.time()
    count2 = df_test.count()
    warm_time = time.time() - start_time

    # Third access (confirm cached)
    start_time = time.time()
    count3 = df_test.count()
    hot_time = time.time() - start_time

    results.append({
        "level": level_name,
        "cold": cold_time,
        "warm": warm_time,
        "hot": hot_time
    })

    print(f"   Cold: {cold_time:.3f}s | Warm: {warm_time:.3f}s | Hot: {hot_time:.3f}s")

    # Cleanup
    df_test.unpersist()

# Summary table
print("\n" + "=" * 60)
print("CACHING STRATEGY SUMMARY")
print("=" * 60)
print(f"{'Level':<20} {'Cold':>10} {'Warm':>10} {'Hot':>10} {'Speedup':>10}")
print("-" * 60)
for r in results:
    speedup = r['cold'] / r['hot'] if r['hot'] > 0 else 0
    print(f"{r['level']:<20} {r['cold']:>10.3f}s {r['warm']:>10.3f}s {r['hot']:>10.3f}s {speedup:>9.1f}x")

### C.3 Spark UI for Optimization Evidence

**How to Access Spark UI:**
1. Spark UI runs at `http://localhost:4040` (default when SparkSession is active)
2. Key tabs to monitor:
   - **Jobs**: Overall job status and timeline
   - **Stages**: Detailed stage execution metrics
   - **Storage**: Cached DataFrames and memory usage
   - **SQL**: Query plans and execution metrics
   - **Executors**: Executor memory and task distribution

**Key Metrics to Monitor:**
- Shuffle read/write sizes
- Task duration distribution
- GC time percentage
- Memory usage per executor

In [ ]:
# =============================================================================
# SPARK UI INFORMATION
# =============================================================================

print("=" * 60)
print("SPARK UI ACCESS INFORMATION")
print("=" * 60)

# Get Spark UI URL
ui_url = spark.sparkContext.uiWebUrl
print(f"\nSpark Web UI: {ui_url}")
print("\nIMPORTANT: Open the URL above in your browser while this notebook is running")
print("           to see real-time job execution and optimization metrics.")

print("\n" + "-" * 60)
print("SPARK UI KEY PAGES:")
print("-" * 60)
print(f"  Jobs:      {ui_url}/jobs/")
print(f"  Stages:    {ui_url}/stages/")
print(f"  Storage:   {ui_url}/storage/")
print(f"  SQL:       {ui_url}/sql/")
print(f"  Executors: {ui_url}/executors/")

# Show current Spark configuration
print("\n" + "-" * 60)
print("CURRENT SPARK CONFIGURATION:")
print("-" * 60)
configs = [
    "spark.sql.shuffle.partitions",
    "spark.default.parallelism",
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.memory.fraction",
    "spark.memory.storageFraction"
]
for config in configs:
    try:
        value = spark.conf.get(config)
        print(f"  {config}: {value}")
    except:
        pass

### C.4 Shuffle Management and Partition Tuning

**Shuffle Operations and Their Cost:**
- `groupBy`, `join`, `distinct`, `repartition` all trigger shuffles
- Shuffles move data across the network (expensive!)
- Goal: Minimize shuffles and optimize partition count

**Partition Tuning Guidelines:**
- Rule of thumb: 2-4 partitions per CPU core
- Each partition should have ~128MB of data
- Too few partitions = underutilization
- Too many partitions = task scheduling overhead

**AQE (Adaptive Query Execution):**
- Dynamically coalesces shuffle partitions
- Handles data skew automatically
- Enables runtime re-optimization

In [ ]:
# =============================================================================
# SHUFFLE MANAGEMENT AND PARTITION TUNING
# =============================================================================

print("=" * 60)
print("PARTITION ANALYSIS AND TUNING")
print("=" * 60)

# Reload data for partition analysis
df_partition = spark.read.parquet(PARTITIONED_DIR)

# Current partition count
print(f"\n1. CURRENT PARTITION STATE:")
print(f"   Number of partitions: {df_partition.rdd.getNumPartitions()}")
print(f"   Configured shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# Analyze partition sizes
def analyze_partitions(df, name="DataFrame"):
    """Analyze partition size distribution"""
    partition_sizes = df.rdd.mapPartitions(
        lambda x: [sum(1 for _ in x)]
    ).collect()

    total = sum(partition_sizes)
    non_empty = [s for s in partition_sizes if s > 0]

    print(f"\n   Partition Analysis for {name}:")
    print(f"   - Total records: {total:,}")
    print(f"   - Partitions: {len(partition_sizes)}")
    print(f"   - Non-empty partitions: {len(non_empty)}")
    if non_empty:
        print(f"   - Min partition size: {min(non_empty):,}")
        print(f"   - Max partition size: {max(non_empty):,}")
        print(f"   - Avg partition size: {sum(non_empty) // len(non_empty):,}")

    return partition_sizes

partition_sizes = analyze_partitions(df_partition, "Raw DataFrame")

In [ ]:
# =============================================================================
# PARTITION OPTIMIZATION COMPARISON
# =============================================================================

print("\n2. PARTITION OPTIMIZATION COMPARISON:")

# Test different partition counts
partition_tests = [2, 4, 8, 16, 32]
partition_results = []

for num_partitions in partition_tests:
    print(f"\n   Testing with {num_partitions} partitions...")

    # Repartition
    df_test = df_partition.repartition(num_partitions)

    # Measure aggregation performance
    start_time = time.time()
    result = df_test.groupBy("borough_clean", "complaint_type") \
        .agg(F.count("*").alias("count")) \
        .orderBy(F.desc("count")) \
        .collect()
    exec_time = time.time() - start_time

    partition_results.append({
        "partitions": num_partitions,
        "time": exec_time,
        "result_count": len(result)
    })
    print(f"   Time: {exec_time:.3f}s")

# Summary
print("\n" + "=" * 60)
print("PARTITION TUNING SUMMARY")
print("=" * 60)
print(f"{'Partitions':>12} {'Time (s)':>12} {'Throughput':>15}")
print("-" * 40)
for r in partition_results:
    throughput = r['result_count'] / r['time'] if r['time'] > 0 else 0
    print(f"{r['partitions']:>12} {r['time']:>12.3f} {throughput:>13.1f}/s")

# Find optimal
best = min(partition_results, key=lambda x: x['time'])
print(f"\nOPTIMAL: {best['partitions']} partitions ({best['time']:.3f}s)")

In [ ]:
# =============================================================================
# SHUFFLE OPERATION ANALYSIS - EXECUTION PLAN REVIEW
# =============================================================================

print("\n3. SHUFFLE OPERATION ANALYSIS:")
print("=" * 60)

# Query that triggers shuffle
df_shuffle_demo = df_partition \
    .groupBy("borough_clean", "agency_clean") \
    .agg(
        F.count("*").alias("count"),
        F.avg("resolution_hours").alias("avg_hours"),
        F.countDistinct("complaint_type").alias("complaint_types")
    ) \
    .orderBy(F.desc("count"))

print("\nEXECUTION PLAN (Physical):")
print("-" * 60)
df_shuffle_demo.explain(mode="simple")

print("\n" + "-" * 60)
print("OPTIMIZED LOGICAL PLAN:")
print("-" * 60)
df_shuffle_demo.explain(mode="extended")

# Force execution and show results
print("\nQUERY RESULTS (Top 15):")
df_shuffle_demo.show(15)

## 2a) PySpark MLlib Implementation

### Data Preparation for ML
Converting 311 service request data into ML-ready features:
- **Target Variable**: Resolution time category (Fast/Medium/Slow)
- **Features**: Borough, agency, complaint type, time features

In [ ]:
# =============================================================================
# ML DATA PREPARATION - FIXED FILTERING
# =============================================================================

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler,
    StandardScaler, Bucketizer, SQLTransformer
)
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier,
    GBTClassifier, DecisionTreeClassifier
)
from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator, BinaryClassificationEvaluator
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql.types import DoubleType
import numpy as np

print("=" * 70)
print("ML DATA PREPARATION")
print("=" * 70)

# Load data from Parquet
df_ml = spark.read.parquet(PARTITIONED_DIR)

# Filter valid records with resolution time
# RELAXED FILTER: Allowed 'UNKNOWN' borough to ensure we have data for the pipeline
df_ml_clean = df_ml.filter(
    (F.col("resolution_hours").isNotNull()) &
    (F.col("resolution_hours") > 0) &
    (F.col("resolution_hours") < 10000) &
    (F.col("agency_clean").isNotNull()) &
    (F.col("complaint_type").isNotNull())
)

# Create target variable: Resolution Time Category
# Fast (<24hrs), Medium (24-72hrs), Slow (>72hrs)
df_ml_prepared = df_ml_clean.withColumn(
    "resolution_category",
    F.when(F.col("resolution_hours") < 24, 0.0)
     .when(F.col("resolution_hours") < 72, 1.0)
     .otherwise(2.0)
).withColumn(
    "resolution_category_binary",
    F.when(F.col("resolution_hours") < 48, 0.0).otherwise(1.0)
)

# Add time-based features
df_ml_prepared = df_ml_prepared.withColumn(
    "is_weekend",
    F.when(F.col("created_day_of_week").isin([1, 7]), 1.0).otherwise(0.0)
).withColumn(
    "is_business_hours",
    F.when((F.col("created_hour") >= 9) & (F.col("created_hour") <= 17), 1.0).otherwise(0.0)
)

# Show data distribution
print("\nTarget Variable Distribution:")
df_ml_prepared.groupBy("resolution_category").count().orderBy("resolution_category").show()

print(f"\nTotal ML Records: {df_ml_prepared.count():,}")

In [ ]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE - UPDATED FOR ROBUSTNESS
# =============================================================================

print("=" * 70)
print("FEATURE ENGINEERING PIPELINE")
print("=" * 70)

# Categorical columns to encode
# We'll focus on agency_clean as borough_clean currently lacks diversity in this sample
categorical_cols = ["agency_clean"]

# Create indexers for categorical columns
# handleInvalid="keep" handles new values in test data
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

# Create one-hot encoders
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_vec")
    for col in categorical_cols
]

# Numerical features
numerical_cols = [
    "created_month", "created_day", "created_hour",
    "created_day_of_week", "is_weekend", "is_business_hours"
]

# Assemble all features into a single vector
feature_cols = [f"{col}_vec" for col in categorical_cols] + numerical_cols
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

# Scale features
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withStd=True,
    withMean=False
)

# Build the pre-processing pipeline
preprocessing_pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

print("Pipeline Stages:")
for i, stage in enumerate(preprocessing_pipeline.getStages()):
    print(f"  {i+1}. {type(stage).__name__}")

# Fit and transform the data
# We ensure we have data before fitting
if df_ml_prepared.count() > 0:
    print("\nFitting preprocessing pipeline...")
    start_time = time.time()
    preprocessing_model = preprocessing_pipeline.fit(df_ml_prepared)
    df_features = preprocessing_model.transform(df_ml_prepared)
    preprocessing_time = time.time() - start_time
    print(f"Preprocessing completed in {preprocessing_time:.2f} seconds")

    # Show sample
    df_features.select("features", "resolution_category").show(5, truncate=False)
else:
    print("\nWARNING: No records found in df_ml_prepared. Please check the filtering in the previous cell.")

In [ ]:
# =============================================================================
# TRAIN/TEST SPLIT
# =============================================================================

print("=" * 70)
print("TRAIN/TEST SPLIT")
print("=" * 70)

# Select relevant columns for ML
df_ml_final = df_features.select(
    "features",
    "resolution_category",
    "resolution_category_binary",
    "resolution_hours"
)

# Cache for multiple model training
df_ml_final.persist(StorageLevel.MEMORY_AND_DISK)

# Split data: 80% train, 20% test
train_df, test_df = df_ml_final.randomSplit([0.8, 0.2], seed=42)

# Cache train and test sets
train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

train_count = train_df.count()
test_count = test_df.count()

print(f"\nTraining Set: {train_count:,} records ({train_count/(train_count+test_count)*100:.1f}%)")
print(f"Test Set: {test_count:,} records ({test_count/(train_count+test_count)*100:.1f}%)")

# Check class balance
print("\nClass Distribution in Training Set:")
train_df.groupBy("resolution_category").count().orderBy("resolution_category").show()

### MLlib Algorithm 1: Logistic Regression
**Distributed Implementation:**
- Uses L-BFGS optimizer distributed across executors
- Supports regularization (L1/L2) for overfitting prevention
- Efficient for large-scale linear classification

In [ ]:
# =============================================================================
# MLLIB ALGORITHM 1: LOGISTIC REGRESSION
# =============================================================================

print("=" * 70)
print("MLLIB ALGORITHM 1: LOGISTIC REGRESSION (Multiclass)")
print("=" * 70)

# Initialize Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="resolution_category",
    predictionCol="prediction_lr",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.5,  # Mix of L1 and L2
    family="multinomial"  # Multiclass
)

# Train model
print("\nTraining Logistic Regression...")
start_time = time.time()
lr_model = lr.fit(train_df)
lr_train_time = time.time() - start_time
print(f"Training completed in {lr_train_time:.2f} seconds")

# Evaluate
lr_predictions = lr_model.transform(test_df)

# Multiclass evaluation
evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="resolution_category",
    predictionCol="prediction_lr",
    metricName="accuracy"
)
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="resolution_category",
    predictionCol="prediction_lr",
    metricName="f1"
)
evaluator_precision = MulticlassClassificationEvaluator(
    labelCol="resolution_category",
    predictionCol="prediction_lr",
    metricName="weightedPrecision"
)

lr_accuracy = evaluator_accuracy.evaluate(lr_predictions)
lr_f1 = evaluator_f1.evaluate(lr_predictions)
lr_precision = evaluator_precision.evaluate(lr_predictions)

print(f"\nLogistic Regression Results:")
print(f"  Accuracy: {lr_accuracy:.4f}")
print(f"  F1 Score: {lr_f1:.4f}")
print(f"  Weighted Precision: {lr_precision:.4f}")
print(f"  Training Time: {lr_train_time:.2f}s")

# Store results for comparison
ml_results = {
    "Logistic Regression (MLlib)": {
        "accuracy": lr_accuracy,
        "f1": lr_f1,
        "precision": lr_precision,
        "train_time": lr_train_time
    }
}

### MLlib Algorithm 2: Random Forest
**Distributed Implementation:**
- Each tree trained on different data partition (bagging)
- Trees trained in parallel across executors
- Feature subsampling at each split
- Robust to overfitting with ensemble approach

In [ ]:
# =============================================================================
# MLLIB ALGORITHM 2: RANDOM FOREST
# =============================================================================

print("=" * 70)
print("MLLIB ALGORITHM 2: RANDOM FOREST")
print("=" * 70)

# Initialize Random Forest
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="resolution_category",
    predictionCol="prediction_rf",
    numTrees=50,
    maxDepth=10,
    maxBins=32,
    featureSubsetStrategy="sqrt",
    seed=42
)

# Train model
print("\nTraining Random Forest (50 trees)...")
start_time = time.time()
rf_model = rf.fit(train_df)
rf_train_time = time.time() - start_time
print(f"Training completed in {rf_train_time:.2f} seconds")

# Evaluate
rf_predictions = rf_model.transform(test_df)

rf_accuracy = evaluator_accuracy.evaluate(
    rf_predictions.withColumn("prediction_lr", F.col("prediction_rf"))
)
rf_f1 = evaluator_f1.evaluate(
    rf_predictions.withColumn("prediction_lr", F.col("prediction_rf"))
)
rf_precision = evaluator_precision.evaluate(
    rf_predictions.withColumn("prediction_lr", F.col("prediction_rf"))
)

print(f"\nRandom Forest Results:")
print(f"  Accuracy: {rf_accuracy:.4f}")
print(f"  F1 Score: {rf_f1:.4f}")
print(f"  Weighted Precision: {rf_precision:.4f}")
print(f"  Training Time: {rf_train_time:.2f}s")
print(f"  Number of Trees: {rf_model.getNumTrees}")

# Feature importance
print("\nTop 10 Feature Importances:")
feature_importance = rf_model.featureImportances.toArray()
for i, importance in enumerate(sorted(enumerate(feature_importance), key=lambda x: -x[1])[:10]):
    print(f"  Feature {importance[0]}: {importance[1]:.4f}")

# Store results
ml_results["Random Forest (MLlib)"] = {
    "accuracy": rf_accuracy,
    "f1": rf_f1,
    "precision": rf_precision,
    "train_time": rf_train_time
}

### MLlib Algorithm 3: Gradient Boosted Trees (GBT)
**Distributed Implementation:**
- Sequential tree building with gradient descent
- Each tree corrects errors of previous trees
- Distributed computation of gradients across partitions
- Often achieves best accuracy among tree-based methods

In [ ]:
# =============================================================================
# MLLIB ALGORITHM 3: GRADIENT BOOSTED TREES (GBT)
# =============================================================================

print("=" * 70)
print("MLLIB ALGORITHM 3: GRADIENT BOOSTED TREES")
print("=" * 70)

# Note: GBTClassifier only supports binary classification
# Using binary target variable for GBT

# Initialize GBT
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="resolution_category_binary",
    predictionCol="prediction_gbt",
    maxIter=30,
    maxDepth=8,
    stepSize=0.1,
    seed=42
)

# Train model
print("\nTraining GBT (30 iterations)...")
start_time = time.time()
gbt_model = gbt.fit(train_df)
gbt_train_time = time.time() - start_time
print(f"Training completed in {gbt_train_time:.2f} seconds")

# Evaluate
gbt_predictions = gbt_model.transform(test_df)

# Binary evaluation
evaluator_binary = BinaryClassificationEvaluator(
    labelCol="resolution_category_binary",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
evaluator_binary_acc = MulticlassClassificationEvaluator(
    labelCol="resolution_category_binary",
    predictionCol="prediction_gbt",
    metricName="accuracy"
)
evaluator_binary_f1 = MulticlassClassificationEvaluator(
    labelCol="resolution_category_binary",
    predictionCol="prediction_gbt",
    metricName="f1"
)

gbt_auc = evaluator_binary.evaluate(gbt_predictions)
gbt_accuracy = evaluator_binary_acc.evaluate(gbt_predictions)
gbt_f1 = evaluator_binary_f1.evaluate(gbt_predictions)

print(f"\nGBT Results (Binary Classification):")
print(f"  AUC-ROC: {gbt_auc:.4f}")
print(f"  Accuracy: {gbt_accuracy:.4f}")
print(f"  F1 Score: {gbt_f1:.4f}")
print(f"  Training Time: {gbt_train_time:.2f}s")
print(f"  Number of Trees: {gbt_model.getNumTrees}")

# Feature importance
print("\nTop 10 Feature Importances (GBT):")
gbt_importance = gbt_model.featureImportances.toArray()
for i, importance in enumerate(sorted(enumerate(gbt_importance), key=lambda x: -x[1])[:10]):
    print(f"  Feature {importance[0]}: {importance[1]:.4f}")

# Store results
ml_results["GBT (MLlib - Binary)"] = {
    "accuracy": gbt_accuracy,
    "f1": gbt_f1,
    "auc": gbt_auc,
    "train_time": gbt_train_time
}

### Scikit-learn Baseline Comparison
Comparing MLlib distributed algorithms with single-node scikit-learn to demonstrate:
- Performance differences
- Training time comparison
- Scalability advantages

In [ ]:
# =============================================================================
# SCIKIT-LEARN BASELINE COMPARISON (Single Node)
# =============================================================================

from sklearn.linear_model import LogisticRegression as SkLogisticRegression
from sklearn.ensemble import RandomForestClassifier as SkRandomForest
from sklearn.ensemble import GradientBoostingClassifier as SkGBT
from sklearn.metrics import accuracy_score, f1_score, precision_score
from sklearn.preprocessing import StandardScaler as SkStandardScaler
import warnings
warnings.filterwarnings('ignore')

print("=" * 70)
print("SCIKIT-LEARN BASELINE COMPARISON (Single Node)")
print("=" * 70)

# Convert Spark DataFrames to Pandas for sklearn
print("\nConverting to Pandas (sampling for sklearn comparison)...")
# Use a sample to make sklearn comparison fair (sklearn is single-node)
sample_size = min(50000, train_count)
sample_ratio = sample_size / train_count

train_pandas = train_df.sample(fraction=sample_ratio, seed=42).toPandas()
test_pandas = test_df.sample(fraction=sample_ratio, seed=42).toPandas()

print(f"Sample size for sklearn: {len(train_pandas):,} train, {len(test_pandas):,} test")

# Extract features and labels
X_train = np.array([row.toArray() for row in train_pandas['features']])
y_train = train_pandas['resolution_category'].values
y_train_binary = train_pandas['resolution_category_binary'].values

X_test = np.array([row.toArray() for row in test_pandas['features']])
y_test = test_pandas['resolution_category'].values
y_test_binary = test_pandas['resolution_category_binary'].values

print(f"Feature dimensions: {X_train.shape}")

In [ ]:
# =============================================================================
# SKLEARN MODEL TRAINING
# =============================================================================

sklearn_results = {}

# 1. Logistic Regression (sklearn)
print("\n1. Sklearn Logistic Regression...")
start_time = time.time()
sk_lr = SkLogisticRegression(max_iter=100, multi_class='multinomial', n_jobs=-1)
sk_lr.fit(X_train, y_train)
sk_lr_time = time.time() - start_time
sk_lr_pred = sk_lr.predict(X_test)
sk_lr_acc = accuracy_score(y_test, sk_lr_pred)
sk_lr_f1 = f1_score(y_test, sk_lr_pred, average='weighted')
print(f"   Accuracy: {sk_lr_acc:.4f}, F1: {sk_lr_f1:.4f}, Time: {sk_lr_time:.2f}s")
sklearn_results["Logistic Regression (sklearn)"] = {
    "accuracy": sk_lr_acc, "f1": sk_lr_f1, "train_time": sk_lr_time
}

# 2. Random Forest (sklearn)
print("\n2. Sklearn Random Forest...")
start_time = time.time()
sk_rf = SkRandomForest(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)
sk_rf.fit(X_train, y_train)
sk_rf_time = time.time() - start_time
sk_rf_pred = sk_rf.predict(X_test)
sk_rf_acc = accuracy_score(y_test, sk_rf_pred)
sk_rf_f1 = f1_score(y_test, sk_rf_pred, average='weighted')
print(f"   Accuracy: {sk_rf_acc:.4f}, F1: {sk_rf_f1:.4f}, Time: {sk_rf_time:.2f}s")
sklearn_results["Random Forest (sklearn)"] = {
    "accuracy": sk_rf_acc, "f1": sk_rf_f1, "train_time": sk_rf_time
}

# 3. GBT (sklearn)
print("\n3. Sklearn GBT...")
start_time = time.time()
sk_gbt = SkGBT(n_estimators=30, max_depth=8, learning_rate=0.1, random_state=42)
sk_gbt.fit(X_train, y_train_binary)
sk_gbt_time = time.time() - start_time
sk_gbt_pred = sk_gbt.predict(X_test)
sk_gbt_acc = accuracy_score(y_test_binary, sk_gbt_pred)
sk_gbt_f1 = f1_score(y_test_binary, sk_gbt_pred, average='weighted')
print(f"   Accuracy: {sk_gbt_acc:.4f}, F1: {sk_gbt_f1:.4f}, Time: {sk_gbt_time:.2f}s")
sklearn_results["GBT (sklearn - Binary)"] = {
    "accuracy": sk_gbt_acc, "f1": sk_gbt_f1, "train_time": sk_gbt_time
}

### Custom Transformer Implementation
Creating domain-specific transformers for 311 service request data:
1. **ServiceUrgencyTransformer**: Calculates urgency score based on complaint type
2. **SpatialClusterTransformer**: Adds geographic cluster features
3. **TemporalPatternTransformer**: Extracts temporal patterns from timestamps

In [ ]:
# =============================================================================
# CUSTOM TRANSFORMER IMPLEMENTATION
# =============================================================================

from pyspark.ml import Transformer
from pyspark.ml.param.shared import HasInputCol, HasOutputCol, Param, Params
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark import keyword_only

print("=" * 70)
print("CUSTOM TRANSFORMER IMPLEMENTATION")
print("=" * 70)

class ServiceUrgencyTransformer(Transformer, HasInputCol, HasOutputCol,
                                 DefaultParamsReadable, DefaultParamsWritable):
    """
    Custom transformer that calculates urgency score based on complaint type.
    Domain-specific feature engineering for 311 service requests.
    """

    # Define urgency mapping parameter
    urgencyMapping = Param(
        Params._dummy(),
        "urgencyMapping",
        "Dictionary mapping complaint types to urgency scores"
    )

    @keyword_only
    def __init__(self, inputCol=None, outputCol=None, urgencyMapping=None):
        super(ServiceUrgencyTransformer, self).__init__()
        # Default urgency mapping for 311 complaint types
        default_mapping = {
            "HEAT/HOT WATER": 1.0,          # High urgency
            "WATER SYSTEM": 0.9,
            "PLUMBING": 0.8,
            "ELECTRIC": 0.85,
            "GAS": 0.95,                     # Very high urgency (safety)
            "NOISE - RESIDENTIAL": 0.3,      # Low urgency
            "NOISE - COMMERCIAL": 0.3,
            "NOISE - STREET/SIDEWALK": 0.3,
            "ILLEGAL PARKING": 0.2,          # Very low urgency
            "STREET CONDITION": 0.5,
            "STREET LIGHT CONDITION": 0.4,
            "GENERAL CONSTRUCTION": 0.4,
            "RODENT": 0.6,
            "UNSANITARY CONDITION": 0.7,
        }
        self._setDefault(urgencyMapping=default_mapping)
        kwargs = self._input_kwargs
        self.setParams(**kwargs)

    @keyword_only
    def setParams(self, inputCol=None, outputCol=None, urgencyMapping=None):
        kwargs = self._input_kwargs
        return self._set(**kwargs)

    def getUrgencyMapping(self):
        return self.getOrDefault(self.urgencyMapping)

    def _transform(self, dataset):
        mapping = self.getUrgencyMapping()
        input_col = self.getInputCol()
        output_col = self.getOutputCol()

        # Create case-when expression for urgency scoring
        urgency_expr = F.lit(0.5)  # Default urgency
        for complaint, score in mapping.items():
            urgency_expr = F.when(
                F.upper(F.col(input_col)).contains(complaint.upper()),
                F.lit(score)
            ).otherwise(urgency_expr)

        return dataset.withColumn(output_col, urgency_expr)

# Test the custom transformer
print("\n1. ServiceUrgencyTransformer:")
urgency_transformer = ServiceUrgencyTransformer(
    inputCol="complaint_type",
    outputCol="urgency_score"
)

# Apply to sample data
df_sample = df_ml_prepared.select("complaint_type").distinct().limit(10)
df_with_urgency = urgency_transformer.transform(df_sample)
df_with_urgency.show(10, truncate=False)

In [ ]:
# =============================================================================
# ADDITIONAL CUSTOM TRANSFORMERS
# =============================================================================

class TemporalPatternTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Custom transformer that extracts temporal patterns from timestamp features.
    Creates features like rush hour indicator, seasonality, etc.
    """

    @keyword_only
    def __init__(self):
        super(TemporalPatternTransformer, self).__init__()

    def _transform(self, dataset):
        return dataset \
            .withColumn(
                "is_rush_hour",
                F.when(
                    ((F.col("created_hour") >= 7) & (F.col("created_hour") <= 9)) |
                    ((F.col("created_hour") >= 16) & (F.col("created_hour") <= 18)),
                    1.0
                ).otherwise(0.0)
            ) \
            .withColumn(
                "is_night",
                F.when(
                    (F.col("created_hour") >= 22) | (F.col("created_hour") <= 6),
                    1.0
                ).otherwise(0.0)
            ) \
            .withColumn(
                "season",
                F.when(F.col("created_month").isin([12, 1, 2]), 0.0)  # Winter
                 .when(F.col("created_month").isin([3, 4, 5]), 1.0)   # Spring
                 .when(F.col("created_month").isin([6, 7, 8]), 2.0)   # Summer
                 .otherwise(3.0)  # Fall
            )

class BoroughDensityTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Custom transformer that adds borough-level complaint density features.
    Simulates lookup join with pre-computed statistics.
    """

    # Pre-computed borough statistics (would normally come from historical data)
    borough_stats = {
        "BROOKLYN": {"density": 0.85, "avg_resolution": 36.5},
        "MANHATTAN": {"density": 0.95, "avg_resolution": 28.3},
        "QUEENS": {"density": 0.70, "avg_resolution": 42.1},
        "BRONX": {"density": 0.75, "avg_resolution": 45.2},
        "STATEN ISLAND": {"density": 0.40, "avg_resolution": 52.8},
    }

    @keyword_only
    def __init__(self):
        super(BoroughDensityTransformer, self).__init__()

    def _transform(self, dataset):
        # Create density and expected resolution columns
        density_expr = F.lit(0.5)
        expected_res_expr = F.lit(40.0)

        for borough, stats in self.borough_stats.items():
            density_expr = F.when(
                F.col("borough_clean") == borough,
                F.lit(stats["density"])
            ).otherwise(density_expr)
            expected_res_expr = F.when(
                F.col("borough_clean") == borough,
                F.lit(stats["avg_resolution"])
            ).otherwise(expected_res_expr)

        return dataset \
            .withColumn("borough_density", density_expr) \
            .withColumn("expected_resolution_hours", expected_res_expr)

# Test custom transformers
print("\n2. TemporalPatternTransformer:")
temporal_transformer = TemporalPatternTransformer()
df_temporal = temporal_transformer.transform(df_ml_prepared.limit(5))
df_temporal.select(
    "created_hour", "created_month", "is_rush_hour", "is_night", "season"
).show()

print("\n3. BoroughDensityTransformer:")
density_transformer = BoroughDensityTransformer()
df_density = density_transformer.transform(df_ml_prepared.limit(5))
df_density.select(
    "borough_clean", "borough_density", "expected_resolution_hours"
).show()

### Model Serialization: MLlib Save/Load and Pickle
**MLlib Native Serialization:**
- Models saved in Parquet format with metadata
- Supports versioning and cross-platform compatibility
- Efficient for large models

**Pickle Serialization:**
- Python-native serialization
- Good for sklearn models and small MLlib models
- Less portable across Spark versions

In [ ]:
# =============================================================================
# MODEL SERIALIZATION
# =============================================================================

import pickle

print("=" * 70)
print("MODEL SERIALIZATION")
print("=" * 70)

MODEL_DIR = "ml_models"

# Create model directory
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

# 1. MLlib Native Save (Parquet-based)
print("\n1. MLLIB NATIVE SERIALIZATION:")
print("-" * 40)

# Save Random Forest model (MLlib)
rf_model_path = f"{MODEL_DIR}/rf_model_mllib"
if os.path.exists(rf_model_path):
    shutil.rmtree(rf_model_path)

start_time = time.time()
rf_model.write().overwrite().save(rf_model_path)
save_time = time.time() - start_time
print(f"Random Forest saved to: {rf_model_path}")
print(f"Save time: {save_time:.2f}s")

# Save GBT model (MLlib)
gbt_model_path = f"{MODEL_DIR}/gbt_model_mllib"
if os.path.exists(gbt_model_path):
    shutil.rmtree(gbt_model_path)

gbt_model.write().overwrite().save(gbt_model_path)
print(f"GBT saved to: {gbt_model_path}")

# Save Logistic Regression model
lr_model_path = f"{MODEL_DIR}/lr_model_mllib"
if os.path.exists(lr_model_path):
    shutil.rmtree(lr_model_path)

lr_model.write().overwrite().save(lr_model_path)
print(f"Logistic Regression saved to: {lr_model_path}")

# Save preprocessing pipeline
pipeline_path = f"{MODEL_DIR}/preprocessing_pipeline"
if os.path.exists(pipeline_path):
    shutil.rmtree(pipeline_path)

preprocessing_model.write().overwrite().save(pipeline_path)
print(f"Preprocessing Pipeline saved to: {pipeline_path}")

In [ ]:
# =============================================================================
# PICKLE SERIALIZATION (for sklearn models)
# =============================================================================

print("\n2. PICKLE SERIALIZATION (sklearn models):")
print("-" * 40)

# Save sklearn models using pickle
sklearn_models = {
    "logistic_regression": sk_lr,
    "random_forest": sk_rf,
    "gbt": sk_gbt
}

for name, model in sklearn_models.items():
    model_path = f"{MODEL_DIR}/{name}_sklearn.pkl"
    start_time = time.time()
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    save_time = time.time() - start_time
    file_size = os.path.getsize(model_path) / 1024  # KB
    print(f"{name}: {model_path} ({file_size:.1f} KB, {save_time:.3f}s)")

# 3. Load and verify models
print("\n3. MODEL LOADING VERIFICATION:")
print("-" * 40)

# Load MLlib models
from pyspark.ml.classification import RandomForestClassificationModel, GBTClassificationModel

loaded_rf = RandomForestClassificationModel.load(rf_model_path)
loaded_gbt = GBTClassificationModel.load(gbt_model_path)
print(f"MLlib Random Forest loaded: {loaded_rf.getNumTrees} trees")
print(f"MLlib GBT loaded: {loaded_gbt.getNumTrees} trees")

# Load sklearn models
with open(f"{MODEL_DIR}/random_forest_sklearn.pkl", 'rb') as f:
    loaded_sk_rf = pickle.load(f)
print(f"Sklearn Random Forest loaded: {loaded_sk_rf.n_estimators} trees")

# Verify loaded model works
test_sample = test_df.limit(10)
loaded_predictions = loaded_rf.transform(test_sample)
print(f"\nVerification: Loaded RF predictions successful!")

# Show model directory structure
print(f"\n4. MODEL DIRECTORY STRUCTURE:")
print("-" * 40)
for item in os.listdir(MODEL_DIR):
    item_path = os.path.join(MODEL_DIR, item)
    if os.path.isdir(item_path):
        size = get_dir_size(item_path) / 1024
        print(f"📁 {item}/ ({size:.1f} KB)")
    else:
        size = os.path.getsize(item_path) / 1024
        print(f"📄 {item} ({size:.1f} KB)")

---
## 2b) Distributed Training & Hyperparameter Tuning

### CrossValidator with Parallelism
**Key Configurations:**
- `parallelism`: Number of models trained in parallel
- `numFolds`: Number of cross-validation folds
- `collectSubModels`: Whether to collect intermediate models

**Parallelism Strategy:**
- Higher parallelism = faster training but more memory
- Optimal: 2-4x number of executor cores

In [ ]:
# =============================================================================
# CROSSVALIDATOR WITH PARALLELISM
# =============================================================================

print("=" * 70)
print("CROSSVALIDATOR WITH PARALLELISM")
print("=" * 70)

# Create a smaller sample for faster cross-validation
cv_sample = train_df.sample(fraction=0.3, seed=42)
cv_sample.persist(StorageLevel.MEMORY_AND_DISK)
cv_count = cv_sample.count()
print(f"Using {cv_count:,} records for cross-validation")

# Define Random Forest for tuning
rf_cv = RandomForestClassifier(
    featuresCol="features",
    labelCol="resolution_category",
    seed=42
)

# Hyperparameter grid with computational constraints
# Grid size = 2 * 2 * 2 = 8 combinations
# With 3-fold CV = 24 total model fits
param_grid = ParamGridBuilder() \
    .addGrid(rf_cv.numTrees, [20, 50]) \
    .addGrid(rf_cv.maxDepth, [5, 10]) \
    .addGrid(rf_cv.maxBins, [16, 32]) \
    .build()

print(f"\nHyperparameter Grid Size: {len(param_grid)} combinations")
print("Grid Parameters:")
print("  - numTrees: [20, 50]")
print("  - maxDepth: [5, 10]")
print("  - maxBins: [16, 32]")

# Evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="resolution_category",
    predictionCol="prediction",
    metricName="f1"
)

# CrossValidator with parallelism
cv = CrossValidator(
    estimator=rf_cv,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=4,  # Train 4 models in parallel
    seed=42
)

print(f"\nCrossValidator Configuration:")
print(f"  - Number of Folds: 3")
print(f"  - Parallelism: 4")
print(f"  - Total Model Fits: {len(param_grid) * 3}")
print(f"  - Metric: F1 Score")

In [ ]:
# =============================================================================
# RUN CROSS-VALIDATION
# =============================================================================

print("\nRunning Cross-Validation...")
print("(This may take several minutes)")
print("-" * 40)

start_time = time.time()
cv_model = cv.fit(cv_sample)
cv_time = time.time() - start_time

print(f"\nCross-Validation completed in {cv_time:.2f} seconds")
print(f"Average time per model: {cv_time / (len(param_grid) * 3):.2f}s")

# Get best model and parameters
best_model = cv_model.bestModel
best_params = best_model.extractParamMap()

print(f"\nBest Model Parameters:")
print(f"  - numTrees: {best_model.getNumTrees}")
print(f"  - maxDepth: {best_model.getOrDefault('maxDepth')}")

# Average metrics for each parameter combination
print(f"\nCross-Validation Metrics (F1 Score):")
print("-" * 40)
avg_metrics = cv_model.avgMetrics
for i, (params, metric) in enumerate(zip(param_grid, avg_metrics)):
    param_str = ", ".join([f"{p.name}={v}" for p, v in params.items()])
    print(f"  {i+1}. {param_str}")
    print(f"     F1 Score: {metric:.4f}")

# Best F1 score
best_idx = avg_metrics.index(max(avg_metrics))
print(f"\nBest Configuration (Index {best_idx + 1}):")
print(f"  F1 Score: {max(avg_metrics):.4f}")

### Model Checkpointing During Training
**Purpose:**
- Save intermediate models during long training jobs
- Enable recovery from failures
- Track training progress over time

**Spark Checkpointing:**
- `spark.sparkContext.setCheckpointDir()` for RDD checkpointing
- Model checkpointing through periodic saves

In [ ]:
# =============================================================================
# MODEL CHECKPOINTING DURING TRAINING
# =============================================================================

print("=" * 70)
print("MODEL CHECKPOINTING")
print("=" * 70)

CHECKPOINT_DIR = "checkpoints"

# Set checkpoint directory for Spark
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR)

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print(f"Checkpoint directory set: {CHECKPOINT_DIR}")

# Custom training loop with checkpointing
def train_with_checkpoints(train_data, test_data, n_iterations, checkpoint_interval):
    """
    Custom training loop that saves checkpoints at regular intervals.
    Simulates iterative training with model saving.
    """
    checkpoints = []
    metrics_history = []

    print(f"\nTraining with checkpoints every {checkpoint_interval} iterations...")

    for i in range(1, n_iterations + 1):
        # Train model with current iteration as max depth
        # (simulating progressive training)
        current_trees = 10 * i

        rf_iter = RandomForestClassifier(
            featuresCol="features",
            labelCol="resolution_category",
            numTrees=current_trees,
            maxDepth=5,
            seed=42
        )

        start_time = time.time()
        model = rf_iter.fit(train_data)
        train_time = time.time() - start_time

        # Evaluate
        predictions = model.transform(test_data)
        f1_score = evaluator.evaluate(predictions)

        metrics_history.append({
            "iteration": i,
            "trees": current_trees,
            "f1": f1_score,
            "time": train_time
        })

        print(f"  Iteration {i}: {current_trees} trees, F1={f1_score:.4f}, Time={train_time:.2f}s")

        # Save checkpoint at intervals
        if i % checkpoint_interval == 0:
            checkpoint_path = f"{CHECKPOINT_DIR}/model_iter_{i}"
            model.write().overwrite().save(checkpoint_path)
            checkpoints.append(checkpoint_path)
            print(f"  Checkpoint saved: {checkpoint_path}")

    return model, checkpoints, metrics_history

# Run training with checkpoints
final_model, checkpoints, history = train_with_checkpoints(
    cv_sample,
    test_df.sample(0.2, seed=42),
    n_iterations=5,
    checkpoint_interval=2
)

print(f"\nCheckpoints created: {len(checkpoints)}")

### Resource Allocation Justification

**Current Configuration Analysis:**
| Resource | Setting | Justification |
|----------|---------|---------------|
| Executors | local[*] | All available cores |
| Driver Memory | 8GB | Large for data collection |
| Executor Memory | 4GB | Balanced for partition size |
| Shuffle Partitions | 8 | 2x expected cores |
| Parallelism | 8 | Matches available threads |

**Memory Formula:**
- Executor Memory = (Data Size / Partitions) × 2-3x overhead
- For 200K records (~50MB): 50/8 × 3 = ~20MB per partition + overhead

In [ ]:
# =============================================================================
# RESOURCE ALLOCATION ANALYSIS
# =============================================================================

print("=" * 70)
print("RESOURCE ALLOCATION ANALYSIS")
print("=" * 70)

# Current Spark configuration
print("\n1. CURRENT SPARK CONFIGURATION:")
print("-" * 40)

config_items = [
    ("spark.driver.memory", "Driver Memory"),
    ("spark.executor.memory", "Executor Memory"),
    ("spark.sql.shuffle.partitions", "Shuffle Partitions"),
    ("spark.default.parallelism", "Default Parallelism"),
    ("spark.memory.fraction", "Memory Fraction"),
    ("spark.memory.storageFraction", "Storage Fraction"),
]

for config, desc in config_items:
    try:
        value = spark.conf.get(config)
        print(f"  {desc}: {value}")
    except:
        print(f"  {desc}: (not set)")

# Calculate optimal resources based on data size
print("\n2. RESOURCE OPTIMIZATION RECOMMENDATIONS:")
print("-" * 40)

data_size_mb = get_dir_size(PARTITIONED_DIR) / (1024 * 1024)
num_cores = spark.sparkContext.defaultParallelism
num_partitions = int(spark.conf.get("spark.sql.shuffle.partitions"))

print(f"  Data Size: {data_size_mb:.2f} MB")
print(f"  Available Cores: {num_cores}")
print(f"  Current Partitions: {num_partitions}")

# Recommendations
optimal_partitions = max(2, int(data_size_mb / 128))  # ~128MB per partition
print(f"\n  Recommendations:")
print(f"  - Optimal Shuffle Partitions: {max(optimal_partitions, num_cores)}")
print(f"  - Optimal Executor Memory: {max(4, int(data_size_mb / 100) + 2)}GB")
print(f"  - Recommended Parallelism for CV: {min(8, num_cores)}")

# Memory usage estimate
print("\n3. MEMORY USAGE ESTIMATE:")
print("-" * 40)
records = train_count + test_count
bytes_per_record = (data_size_mb * 1024 * 1024) / records
print(f"  Records: {records:,}")
print(f"  Bytes per Record: ~{bytes_per_record:.0f} bytes")
print(f"  Estimated Memory (cached): {(records * bytes_per_record * 3) / (1024**3):.2f} GB")
print(f"  (with 3x overhead for Spark operations)")

---
## 2c) Scalability Analysis

### Strong Scaling vs Weak Scaling
**Strong Scaling:**
- Fixed problem size, increasing resources
- Measures: Speedup = T(1) / T(N)
- Ideal: Linear speedup (2x resources = 2x faster)

**Weak Scaling:**
- Problem size grows proportionally with resources
- Measures: Efficiency = T(1) / T(N)
- Ideal: Constant time regardless of resources/data

In [ ]:
# =============================================================================
# STRONG SCALING ANALYSIS
# =============================================================================

print("=" * 70)
print("STRONG SCALING ANALYSIS")
print("=" * 70)
print("Fixed problem size, varying partition count (simulating resources)")

# Use fixed dataset
fixed_data = train_df.sample(fraction=0.5, seed=42)
fixed_count = fixed_data.count()
print(f"\nFixed Dataset Size: {fixed_count:,} records")

# Test with different partition counts (simulating different resource levels)
partition_configs = [2, 4, 8, 16]
strong_scaling_results = []

for num_parts in partition_configs:
    print(f"\nTesting with {num_parts} partitions...")

    # Repartition data
    data_repartitioned = fixed_data.repartition(num_parts)
    data_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)
    data_repartitioned.count()  # Force materialization

    # Train model
    rf_scale = RandomForestClassifier(
        featuresCol="features",
        labelCol="resolution_category",
        numTrees=20,
        maxDepth=5,
        seed=42
    )

    start_time = time.time()
    model = rf_scale.fit(data_repartitioned)
    train_time = time.time() - start_time

    strong_scaling_results.append({
        "partitions": num_parts,
        "time": train_time,
        "throughput": fixed_count / train_time
    })

    print(f"  Time: {train_time:.2f}s, Throughput: {fixed_count / train_time:.0f} rec/s")

    data_repartitioned.unpersist()

# Calculate speedup
base_time = strong_scaling_results[0]["time"]
print("\n" + "-" * 70)
print("STRONG SCALING RESULTS:")
print("-" * 70)
print(f"{'Partitions':>12} {'Time (s)':>12} {'Speedup':>12} {'Efficiency':>12}")
print("-" * 50)
for result in strong_scaling_results:
    speedup = base_time / result["time"]
    ideal_speedup = result["partitions"] / partition_configs[0]
    efficiency = (speedup / ideal_speedup) * 100
    print(f"{result['partitions']:>12} {result['time']:>12.2f} {speedup:>12.2f}x {efficiency:>11.1f}%")

In [ ]:
# =============================================================================
# WEAK SCALING ANALYSIS
# =============================================================================

print("\n" + "=" * 70)
print("WEAK SCALING ANALYSIS")
print("=" * 70)
print("Problem size grows with partition count (constant work per partition)")

# Base records per partition
base_records_per_partition = 5000
weak_scaling_results = []

partition_configs_weak = [2, 4, 8]

for num_parts in partition_configs_weak:
    target_size = base_records_per_partition * num_parts
    sample_fraction = min(1.0, target_size / train_count)

    print(f"\nPartitions: {num_parts}, Target records: {target_size:,}")

    # Sample proportional data
    data_sampled = train_df.sample(fraction=sample_fraction, seed=42)
    actual_count = data_sampled.count()
    print(f"  Actual records: {actual_count:,}")

    # Repartition
    data_repartitioned = data_sampled.repartition(num_parts)
    data_repartitioned.persist(StorageLevel.MEMORY_AND_DISK)
    data_repartitioned.count()

    # Train model
    rf_scale = RandomForestClassifier(
        featuresCol="features",
        labelCol="resolution_category",
        numTrees=20,
        maxDepth=5,
        seed=42
    )

    start_time = time.time()
    model = rf_scale.fit(data_repartitioned)
    train_time = time.time() - start_time

    weak_scaling_results.append({
        "partitions": num_parts,
        "records": actual_count,
        "time": train_time,
        "records_per_partition": actual_count / num_parts
    })

    print(f"  Time: {train_time:.2f}s")

    data_repartitioned.unpersist()

# Calculate weak scaling efficiency
base_time = weak_scaling_results[0]["time"]
print("\n" + "-" * 70)
print("WEAK SCALING RESULTS:")
print("-" * 70)
print(f"{'Partitions':>12} {'Records':>12} {'Time (s)':>12} {'Efficiency':>12}")
print("-" * 50)
for result in weak_scaling_results:
    efficiency = (base_time / result["time"]) * 100
    print(f"{result['partitions']:>12} {result['records']:>12,} {result['time']:>12.2f} {efficiency:>11.1f}%")

print("\nInterpretation:")
print("- Efficiency >100%: Super-linear scaling (rare)")
print("- Efficiency ~100%: Ideal weak scaling")
print("- Efficiency <100%: Communication/coordination overhead")

### Bottleneck Identification
**Common Bottlenecks in Distributed ML:**

| Bottleneck | Symptoms | Solutions |
|------------|----------|-----------|
| **I/O Bound** | High disk wait, slow reads | Parquet, caching, partitioning |
| **Network Bound** | High shuffle time | Broadcast joins, reduce shuffles |
| **Computation Bound** | High CPU usage | More executors, vectorization |
| **Memory Bound** | OOM errors, spilling | Increase memory, reduce partitions |
| **Data Skew** | Uneven task times | Salting, repartition |

In [ ]:
# =============================================================================
# BOTTLENECK IDENTIFICATION
# =============================================================================

print("=" * 70)
print("BOTTLENECK IDENTIFICATION")
print("=" * 70)

# Simulate and measure different operations to identify bottlenecks
bottleneck_results = {}

# 1. I/O Bottleneck Test (Read operations)
print("\n1. I/O PERFORMANCE (Read from Parquet):")
print("-" * 40)

# Cold read
start_time = time.time()
df_io_test = spark.read.parquet(PARTITIONED_DIR)
cold_count = df_io_test.count()
cold_read_time = time.time() - start_time
print(f"  Cold Read: {cold_read_time:.3f}s ({cold_count:,} records)")

# Warm read (cached)
df_io_test.persist(StorageLevel.MEMORY_AND_DISK)
df_io_test.count()  # Materialize cache
start_time = time.time()
warm_count = df_io_test.count()
warm_read_time = time.time() - start_time
print(f"  Warm Read (cached): {warm_read_time:.3f}s ({warm_count:,} records)")
print(f"  Cache Speedup: {cold_read_time / warm_read_time:.1f}x")
bottleneck_results["I/O"] = {"cold": cold_read_time, "warm": warm_read_time}
df_io_test.unpersist()

# 2. Computation Bottleneck Test (CPU-intensive transformations)
print("\n2. COMPUTATION PERFORMANCE:")
print("-" * 40)

df_compute = spark.read.parquet(PARTITIONED_DIR)
df_compute.persist(StorageLevel.MEMORY_AND_DISK)
df_compute.count()

start_time = time.time()
# CPU-intensive operations
df_heavy = df_compute \
    .withColumn("sqrt_hours", F.sqrt("resolution_hours")) \
    .withColumn("log_hours", F.log(F.col("resolution_hours") + 1)) \
    .withColumn("sin_hour", F.sin(F.col("created_hour"))) \
    .withColumn("complex", F.sqrt(F.col("sqrt_hours") * F.col("log_hours")))
compute_count = df_heavy.count()
compute_time = time.time() - start_time
print(f"  Complex Transformation: {compute_time:.3f}s")
bottleneck_results["Computation"] = compute_time
df_compute.unpersist()

# 3. Shuffle Bottleneck Test (Network-intensive operations)
print("\n3. SHUFFLE/NETWORK PERFORMANCE:")
print("-" * 40)

df_shuffle = spark.read.parquet(PARTITIONED_DIR)

start_time = time.time()
# Shuffle-heavy operations
df_shuffled = df_shuffle \
    .groupBy("borough_clean", "agency_clean", "complaint_type") \
    .agg(
        F.count("*").alias("count"),
        F.avg("resolution_hours").alias("avg_hours"),
        F.stddev("resolution_hours").alias("std_hours")
    ) \
    .orderBy(F.desc("count"))
shuffle_count = df_shuffled.count()
shuffle_time = time.time() - start_time
print(f"  GroupBy + Aggregation: {shuffle_time:.3f}s ({shuffle_count} groups)")
bottleneck_results["Shuffle"] = shuffle_time

In [ ]:
# =============================================================================
# DATA SKEW ANALYSIS
# =============================================================================

print("\n4. DATA SKEW ANALYSIS:")
print("-" * 40)

# Check partition size distribution
df_skew = spark.read.parquet(PARTITIONED_DIR)
partition_sizes = df_skew.rdd.mapPartitions(
    lambda x: [sum(1 for _ in x)]
).collect()

min_size = min(partition_sizes)
max_size = max(partition_sizes)
avg_size = sum(partition_sizes) / len(partition_sizes)
skew_ratio = max_size / min_size if min_size > 0 else float('inf')

print(f"  Partitions: {len(partition_sizes)}")
print(f"  Min partition size: {min_size:,}")
print(f"  Max partition size: {max_size:,}")
print(f"  Avg partition size: {avg_size:,.0f}")
print(f"  Skew ratio (max/min): {skew_ratio:.2f}")

if skew_ratio > 3:
    print("  WARNING: Significant data skew detected!")
    print("      Consider repartitioning or using salting.")
else:
    print("  Data skew is within acceptable range")

# 5. BOTTLENECK SUMMARY
print("\n" + "=" * 70)
print("BOTTLENECK SUMMARY")
print("=" * 70)

print("\nOperation Times:")
print(f"  I/O (cold read):    {bottleneck_results['I/O']['cold']:.3f}s")
print(f"  I/O (cached read):  {bottleneck_results['I/O']['warm']:.3f}s")
print(f"  Computation:        {bottleneck_results['Computation']:.3f}s")
print(f"  Shuffle/Network:    {bottleneck_results['Shuffle']:.3f}s")

# Identify primary bottleneck
times = {
    "I/O": bottleneck_results['I/O']['cold'],
    "Computation": bottleneck_results['Computation'],
    "Shuffle": bottleneck_results['Shuffle']
}
primary_bottleneck = max(times, key=times.get)
print(f"\nPrimary Bottleneck: {primary_bottleneck}")

recommendations = {
    "I/O": "Use Parquet, enable caching, optimize partition scheme",
    "Computation": "Add more executors, use vectorized operations",
    "Shuffle": "Use broadcast joins, reduce groupBy operations, tune partitions"
}
print(f"Recommendation: {recommendations[primary_bottleneck]}")

### Cost-Performance Tradeoff Analysis
**Cloud Cost Model (AWS EMR Example):**
- m5.xlarge: 4 vCPU, 16GB RAM, $0.192/hr
- m5.2xlarge: 8 vCPU, 32GB RAM, $0.384/hr
- m5.4xlarge: 16 vCPU, 64GB RAM, $0.768/hr

**Tradeoff Considerations:**
- More resources = faster training but higher cost
- Optimal point balances time-to-result with budget
- Consider elasticity: scale up for training, down for inference

In [ ]:
# =============================================================================
# COST-PERFORMANCE TRADEOFF ANALYSIS
# =============================================================================

print("=" * 70)
print("COST-PERFORMANCE TRADEOFF ANALYSIS")
print("=" * 70)

# Define cloud instance types (AWS EMR pricing example)
instance_types = {
    "m5.xlarge": {"vcpu": 4, "memory_gb": 16, "cost_per_hour": 0.192},
    "m5.2xlarge": {"vcpu": 8, "memory_gb": 32, "cost_per_hour": 0.384},
    "m5.4xlarge": {"vcpu": 16, "memory_gb": 64, "cost_per_hour": 0.768},
    "m5.8xlarge": {"vcpu": 32, "memory_gb": 128, "cost_per_hour": 1.536},
}

# Simulate performance based on strong scaling results
# Assume baseline is m5.xlarge (4 cores)
baseline_time_minutes = 10  # Hypothetical training time
baseline_cores = 4

cost_analysis = []

print("\nSIMULATED COST-PERFORMANCE ANALYSIS:")
print("-" * 70)
print(f"{'Instance':<15} {'vCPUs':>8} {'Memory':>10} {'Est. Time':>12} {'Cost/hr':>10} {'Total Cost':>12}")
print("-" * 70)

for instance, specs in instance_types.items():
    # Estimate time based on strong scaling (assume 80% efficiency)
    scaling_factor = specs["vcpu"] / baseline_cores
    efficiency = 0.80  # Account for overhead
    estimated_time = baseline_time_minutes / (scaling_factor * efficiency)

    # Calculate cost
    hours = estimated_time / 60
    total_cost = hours * specs["cost_per_hour"]

    cost_analysis.append({
        "instance": instance,
        "vcpu": specs["vcpu"],
        "memory": specs["memory_gb"],
        "time_min": estimated_time,
        "cost_per_hour": specs["cost_per_hour"],
        "total_cost": total_cost
    })

    print(f"{instance:<15} {specs['vcpu']:>8} {specs['memory_gb']:>8} GB {estimated_time:>10.2f} min ${specs['cost_per_hour']:>9.3f} ${total_cost:>11.4f}")

# Find optimal configuration
cheapest = min(cost_analysis, key=lambda x: x["total_cost"])
fastest = min(cost_analysis, key=lambda x: x["time_min"])

print("\n" + "-" * 70)
print(f"CHEAPEST: {cheapest['instance']} (${cheapest['total_cost']:.4f})")
print(f"FASTEST:  {fastest['instance']} ({fastest['time_min']:.2f} min)")

# Cost efficiency metric: performance per dollar
print("\nCOST EFFICIENCY (lower is better):")
print("-" * 40)
for config in cost_analysis:
    efficiency = config["total_cost"] / config["vcpu"]  # Cost per vCPU
    print(f"  {config['instance']}: ${efficiency:.4f} per vCPU")

In [ ]:
# =============================================================================
# SCALABILITY VISUALIZATION
# =============================================================================

import matplotlib.pyplot as plt

print("=" * 70)
print("SCALABILITY VISUALIZATION")
print("=" * 70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Strong Scaling Plot
ax1 = axes[0, 0]
partitions = [r["partitions"] for r in strong_scaling_results]
times = [r["time"] for r in strong_scaling_results]
ideal_times = [strong_scaling_results[0]["time"] / (p / partitions[0]) for p in partitions]

ax1.plot(partitions, times, 'bo-', label='Actual', linewidth=2, markersize=8)
ax1.plot(partitions, ideal_times, 'r--', label='Ideal', linewidth=2)
ax1.set_xlabel('Partitions (Resources)', fontsize=12)
ax1.set_ylabel('Time (seconds)', fontsize=12)
ax1.set_title('Strong Scaling: Fixed Problem Size', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Weak Scaling Plot
ax2 = axes[0, 1]
partitions_weak = [r["partitions"] for r in weak_scaling_results]
times_weak = [r["time"] for r in weak_scaling_results]
ideal_weak = [weak_scaling_results[0]["time"]] * len(partitions_weak)

ax2.plot(partitions_weak, times_weak, 'go-', label='Actual', linewidth=2, markersize=8)
ax2.plot(partitions_weak, ideal_weak, 'r--', label='Ideal (constant)', linewidth=2)
ax2.set_xlabel('Partitions (Resources)', fontsize=12)
ax2.set_ylabel('Time (seconds)', fontsize=12)
ax2.set_title('Weak Scaling: Proportional Problem Size', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Bottleneck Comparison
ax3 = axes[1, 0]
bottleneck_names = ['I/O (cold)', 'I/O (cached)', 'Computation', 'Shuffle']
bottleneck_times = [
    bottleneck_results['I/O']['cold'],
    bottleneck_results['I/O']['warm'],
    bottleneck_results['Computation'],
    bottleneck_results['Shuffle']
]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1', '#96ceb4']
ax3.barh(bottleneck_names, bottleneck_times, color=colors)
ax3.set_xlabel('Time (seconds)', fontsize=12)
ax3.set_title('Bottleneck Analysis', fontsize=14)
ax3.grid(True, alpha=0.3, axis='x')

# 4. Cost-Performance Tradeoff
ax4 = axes[1, 1]
instances = [c["instance"] for c in cost_analysis]
costs = [c["total_cost"] for c in cost_analysis]
times_cost = [c["time_min"] for c in cost_analysis]

ax4.scatter(times_cost, costs, s=200, c=range(len(cost_analysis)), cmap='viridis')
for i, inst in enumerate(instances):
    ax4.annotate(inst, (times_cost[i], costs[i]), textcoords="offset points",
                 xytext=(0,10), ha='center')
ax4.set_xlabel('Training Time (minutes)', fontsize=12)
ax4.set_ylabel('Total Cost ($)', fontsize=12)
ax4.set_title('Cost-Performance Tradeoff', fontsize=14)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('scalability_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nVisualization saved to: scalability_analysis.png")

In [ ]:
# =============================================================================
# FINAL CLEANUP AND SUMMARY
# =============================================================================

print("=" * 70)
print("PART 2 EXECUTION SUMMARY")
print("=" * 70)

# Unpersist all cached DataFrames
print("\nCleaning up cached DataFrames...")
train_df.unpersist()
test_df.unpersist()
df_ml_final.unpersist()
cv_sample.unpersist()
print("All caches cleared")

In [ ]:
# =============================================================================
# TABLEAU VISUALIZATION — SETUP & DATA EXPORT DIRECTORY
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import pandas as pd
import numpy as np
import warnings, os, json
warnings.filterwarnings('ignore')

# Tableau-aligned colour palette
TABLEAU_BLUE   = '#4E79A7'
TABLEAU_ORANGE = '#F28E2B'
TABLEAU_RED    = '#E15759'
TABLEAU_TEAL   = '#76B7B2'
TABLEAU_GREEN  = '#59A14F'
TABLEAU_YELLOW = '#EDC948'
TABLEAU_PURPLE = '#B07AA1'
TABLEAU_PINK   = '#FF9DA7'
TABLEAU_BG     = '#F5F5F5'

PALETTE = [TABLEAU_BLUE, TABLEAU_ORANGE, TABLEAU_RED,
           TABLEAU_TEAL, TABLEAU_GREEN, TABLEAU_YELLOW, TABLEAU_PURPLE]

sns.set_theme(style="whitegrid", palette=PALETTE)

# Output directory for all Tableau extracts and images
TABLEAU_DIR = "tableau_exports"
os.makedirs(TABLEAU_DIR, exist_ok=True)

print("=" * 65)
print("TABLEAU VISUALIZATION PIPELINE")
print("=" * 65)
print(f"Export directory : {TABLEAU_DIR}/")
print(f"Colour palette   : Tableau 10 (accessible)")
print(f"Dashboards       : 4  |  Views per dashboard: 4-6")

In [ ]:
# =============================================================================
# PREPARE ALL DASHBOARD DATASETS FROM PARQUET - FIXED COLUMN REFERENCES
# =============================================================================

print("Loading data from Parquet for all dashboards …")
start_time = time.time()

df_viz = spark.read.parquet(PARTITIONED_DIR)

# ── (A) DATA QUALITY ────────────────────────────────────────────────────────
# Null rates per column - Using only existing columns
key_cols = [
    "unique_key","borough_clean","agency_clean","complaint_type",
    "Status","resolution_hours","created_date_parsed",
    "Latitude","Longitude"
]
total = df_viz.count()

null_rows = []
for c in key_cols:
    n = df_viz.filter(F.col(c).isNull()).count()
    null_rows.append({"column": c, "null_count": n,
                      "null_pct": round(n / total * 100, 2),
                      "completeness": round((1 - n / total) * 100, 2)})
dq_nulls = pd.DataFrame(null_rows)

# Partition stats
part_sizes = df_viz.rdd.mapPartitions(lambda it: [sum(1 for _ in it)]).collect()
dq_partitions = pd.DataFrame({
    "partition": list(range(len(part_sizes))),
    "records":   part_sizes
})

# Ingestion volume by month
ingestion_vol = df_viz.groupBy("created_month","created_year") \
    .count().orderBy("created_year","created_month").toPandas()
ingestion_vol["period"] = (
    ingestion_vol["created_year"].astype(str) + "-" +
    ingestion_vol["created_month"].astype(str).str.zfill(2)
)

# ── (B) MODEL PERFORMANCE ───────────────────────────────────────────────────
model_metrics = pd.DataFrame([
    {"Model":"Logistic Regression (MLlib)","Accuracy":ml_results["Logistic Regression (MLlib)"]["accuracy"],
     "F1":ml_results["Logistic Regression (MLlib)"]["f1"],"Type":"Distributed"},
    {"Model":"Random Forest (MLlib)",       "Accuracy":ml_results["Random Forest (MLlib)"]["accuracy"],
     "F1":ml_results["Random Forest (MLlib)"]["f1"],"Type":"Distributed"},
    {"Model":"GBT (MLlib)",                 "Accuracy":ml_results["GBT (MLlib - Binary)"]["accuracy"],
     "F1":ml_results["GBT (MLlib - Binary)"]["f1"],"Type":"Distributed"},
    {"Model":"Logistic Regression (sklearn)","Accuracy":sklearn_results["Logistic Regression (sklearn)"]["accuracy"],
     "F1":sklearn_results["Logistic Regression (sklearn)"]["f1"],"Type":"Single-Node"},
    {"Model":"Random Forest (sklearn)",      "Accuracy":sklearn_results["Random Forest (sklearn)"]["accuracy"],
     "F1":sklearn_results["Random Forest (sklearn)"]["f1"],"Type":"Single-Node"},
    {"Model":"GBT (sklearn)",                "Accuracy":sklearn_results["GBT (sklearn - Binary)"]["accuracy"],
     "F1":sklearn_results["GBT (sklearn - Binary)"]["f1"],"Type":"Single-Node"},
])

train_times = pd.DataFrame([
    {"Model":"Logistic Regression","framework":"MLlib","time_s":ml_results["Logistic Regression (MLlib)"]["train_time"], "label": "LR (MLlib)"},
    {"Model":"Random Forest","framework":"MLlib","time_s":ml_results["Random Forest (MLlib)"]["train_time"], "label": "RF (MLlib)"},
    {"Model":"GBT","framework":"MLlib","time_s":ml_results["GBT (MLlib - Binary)"]["train_time"], "label": "GBT (MLlib)"},
    {"Model":"Logistic Regression","framework":"sklearn","time_s":sklearn_results["Logistic Regression (sklearn)"]["train_time"], "label": "LR (sk)"},
    {"Model":"Random Forest","framework":"sklearn","time_s":sklearn_results["Random Forest (sklearn)"]["train_time"], "label": "RF (sk)"},
    {"Model":"GBT","framework":"sklearn","time_s":sklearn_results["GBT (sklearn - Binary)"]["train_time"], "label": "GBT (sk)"},
])

feat_imp = pd.DataFrame({
    "feature": [f"Feature_{i}" for i in range(len(rf_model.featureImportances))],
    "importance": rf_model.featureImportances.toArray()
}).sort_values("importance", ascending=False).head(15).reset_index(drop=True)

# ── (C) BUSINESS INSIGHTS ──────────────────────────────────────────────────
borough_perf = df_viz.groupBy("borough_clean").agg(
    F.count("*").alias("total_complaints"),
    F.avg("resolution_hours").alias("avg_resolution_hours"),
    F.sum(F.when(F.col("Status") == "Closed", 1).otherwise(0)).alias("closed"),
).withColumn("resolution_rate", F.round(F.col("closed")/F.col("total_complaints")*100,2)
).withColumnRenamed("borough_clean", "borough") \
.orderBy(F.desc("total_complaints")).toPandas()

top_complaints = df_viz.groupBy("complaint_type").count() \
    .orderBy(F.desc("count")).limit(15).toPandas()

agency_perf = df_viz.groupBy("agency_clean").agg(
    F.count("*").alias("complaint_count"),
    F.avg("resolution_hours").alias("avg_resolution_hours"),
).withColumnRenamed("agency_clean", "agency") \
.orderBy(F.desc("complaint_count")).limit(12).toPandas()

# Channel distribution - Fallback since column missing
channel_dist = pd.DataFrame([{"channel": "Unknown", "count": total}])

hourly_pattern = df_viz.groupBy("created_hour").count() \
    .withColumnRenamed("created_hour", "hour_of_day") \
    .orderBy("hour_of_day").toPandas()

monthly_trend = df_viz.groupBy("created_month","created_year").agg(
    F.count("*").alias("count"),
    F.avg("resolution_hours").alias("avg_resolution_hours")
).orderBy("created_year","created_month").toPandas()
monthly_trend["period"] = monthly_trend["created_year"].astype(str) + "-" + monthly_trend["created_month"].astype(str)

# ── (D) SCALABILITY & COST ─────────────────────────────────────────────────
strong_sc_df = pd.DataFrame(strong_scaling_results)
# Add efficiency for plot
strong_sc_df['speedup'] = strong_sc_df['time'].iloc[0] / strong_sc_df['time']
strong_sc_df['efficiency_pct'] = (strong_sc_df['speedup'] / (strong_sc_df['partitions'] / strong_sc_df['partitions'].iloc[0])) * 100

weak_sc_df   = pd.DataFrame(weak_scaling_results)
cost_df      = pd.DataFrame(cost_analysis)
# Add records per cost for bubble chart
cost_df['cost_per_record'] = cost_df['total_cost'] / (train_count + test_count)
cost_df['throughput'] = (train_count + test_count) / (cost_df['time_min'] * 60)

bottleneck_df = pd.DataFrame([
    {"operation":"I/O Cold Read",    "duration_s": bottleneck_results["I/O"]["cold"],   "Category":"I/O"},
    {"operation":"I/O Cached Read",  "duration_s": bottleneck_results["I/O"]["warm"],   "Category":"I/O"},
    {"operation":"Computation",      "duration_s": bottleneck_results["Computation"],   "Category":"CPU"},
    {"operation":"Shuffle/Network",  "duration_s": bottleneck_results["Shuffle"],       "Category":"Network"},
])

elapsed = time.time() - start_time
print(f"All datasets prepared in {elapsed:.2f}s")

## 3a — Dashboard 1: Data Quality & Pipeline Monitoring

**Purpose**: Give data engineers real-time visibility into pipeline health.

**Views included:**
1. **Completeness Heatmap** — % complete per column (LOD: FIXED)
2. **Partition Size Bar** — record distribution across Spark partitions
3. **Ingestion Volume Timeline** — complaint volume over time
4. **Null Rate Gauge** — overall pipeline quality score

**Tableau Extract**: `tableau_exports/dash1_data_quality.csv`

In [ ]:
# =============================================================================
# DASHBOARD 1 — DATA QUALITY & PIPELINE MONITORING
# =============================================================================

# ── Export Tableau Extract ──────────────────────────────────────────────────
dq_nulls.to_csv(f"{TABLEAU_DIR}/dash1_data_quality.csv", index=False)
dq_partitions.to_csv(f"{TABLEAU_DIR}/dash1_partitions.csv", index=False)
ingestion_vol.to_csv(f"{TABLEAU_DIR}/dash1_ingestion_volume.csv", index=False)

overall_quality = dq_nulls["completeness"].mean()

# ── Dashboard Layout ────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12), facecolor=TABLEAU_BG)
fig.suptitle("Dashboard 1 · Data Quality & Pipeline Monitoring",
             fontsize=18, fontweight="bold", color="#333333", y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1-A  KPI Gauge ─ overall quality score
ax_kpi = fig.add_subplot(gs[0, 0])
theta = np.linspace(0, np.pi, 200)
r_outer, r_inner = 1.0, 0.55
ax_kpi.fill_between(np.cos(theta)*r_outer, np.sin(theta)*r_outer,
                    np.cos(theta)*r_inner, np.sin(theta)*r_inner,
                    alpha=0.15, color=TABLEAU_RED)
# colour arc by score
needle_ang = np.pi * (1 - overall_quality / 100)
ax_kpi.annotate("", xy=(np.cos(needle_ang)*0.8, np.sin(needle_ang)*0.8),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=TABLEAU_BLUE, lw=3))
ax_kpi.text(0, -0.15, f"{overall_quality:.1f}%", ha="center", fontsize=20,
            fontweight="bold", color=TABLEAU_BLUE)
ax_kpi.text(0, -0.38, "Overall Completeness", ha="center", fontsize=9, color="#555")
ax_kpi.set_xlim(-1.2, 1.2); ax_kpi.set_ylim(-0.5, 1.2)
ax_kpi.set_aspect("equal"); ax_kpi.axis("off")
ax_kpi.set_title("Pipeline Quality Score", fontsize=11, pad=6)

# 1-B  Completeness horizontal bar
ax_bar = fig.add_subplot(gs[0, 1:])
colors_bar = [TABLEAU_GREEN if v >= 90 else TABLEAU_ORANGE if v >= 70 else TABLEAU_RED
              for v in dq_nulls["completeness"]]
ax_bar.barh(dq_nulls["column"], dq_nulls["completeness"], color=colors_bar, edgecolor="white")
ax_bar.axvline(90, color=TABLEAU_GREEN, ls="--", lw=1.2, label="90% threshold")
ax_bar.axvline(70, color=TABLEAU_ORANGE, ls="--", lw=1.2, label="70% threshold")
for i, (v, n) in enumerate(zip(dq_nulls["completeness"], dq_nulls["null_count"])):
    ax_bar.text(v + 0.3, i, f"{v:.1f}%  (↓ {n:,})", va="center", fontsize=8)
ax_bar.set_xlim(0, 112); ax_bar.set_xlabel("% Completeness")
ax_bar.set_title("Column Completeness (LOD: FIXED [Column])", fontsize=11)
ax_bar.legend(fontsize=8, loc="lower right")

# 1-C  Partition size distribution
ax_part = fig.add_subplot(gs[1, 0])
ax_part.bar(dq_partitions["partition"], dq_partitions["records"],
            color=TABLEAU_TEAL, edgecolor="white")
ax_part.axhline(dq_partitions["records"].mean(), color=TABLEAU_RED,
                ls="--", lw=1.5, label=f"Mean={dq_partitions['records'].mean():.0f}")
ax_part.set_xlabel("Partition ID"); ax_part.set_ylabel("Records")
ax_part.set_title("Spark Partition Distribution\n(Data Skew Check)", fontsize=11)
ax_part.legend(fontsize=8)

# 1-D  Null rate bubble chart
ax_null = fig.add_subplot(gs[1, 1])
x_pos = range(len(dq_nulls))
sizes = [max(20, n / total * 5000) for n in dq_nulls["null_count"]]
sc = ax_null.scatter(x_pos, dq_nulls["null_pct"], s=sizes,
                     c=dq_nulls["null_pct"], cmap="RdYlGn_r", alpha=0.8,
                     edgecolors="grey", linewidths=0.5)
plt.colorbar(sc, ax=ax_null, label="Null %", shrink=0.8)
ax_null.set_xticks(list(x_pos))
ax_null.set_xticklabels(dq_nulls["column"], rotation=45, ha="right", fontsize=7)
ax_null.set_ylabel("Null Rate (%)"); ax_null.set_title("Null Rate Bubble Chart", fontsize=11)

# 1-E  Monthly ingestion volume
ax_vol = fig.add_subplot(gs[1, 2])
if not ingestion_vol.empty:
    ax_vol.plot(range(len(ingestion_vol)), ingestion_vol["count"],
                color=TABLEAU_BLUE, lw=2, marker="o", markersize=4)
    ax_vol.fill_between(range(len(ingestion_vol)), ingestion_vol["count"],
                        alpha=0.15, color=TABLEAU_BLUE)
    ax_vol.set_xticks(range(len(ingestion_vol)))
    ax_vol.set_xticklabels(ingestion_vol["period"], rotation=45, ha="right", fontsize=7)
ax_vol.set_ylabel("Records Ingested")
ax_vol.set_title("Ingestion Volume Timeline", fontsize=11)

plt.savefig(f"{TABLEAU_DIR}/dashboard1_data_quality.png", dpi=150,
            bbox_inches="tight", facecolor=TABLEAU_BG)
plt.show()
print(f"  Saved: {TABLEAU_DIR}/dashboard1_data_quality.png")
print(f"  Extracts: dash1_data_quality.csv, dash1_partitions.csv, dash1_ingestion_volume.csv")

## 3a — Dashboard 2: Model Performance & Feature Analysis

| View | Chart Type | Key Fields |
|------|-----------|------------|
| Accuracy & F1 Comparison | Grouped Bar Chart | model, accuracy, f1 |
| Training Time (MLlib vs sklearn) | Grouped Bar (log scale) | framework, model, duration_s |
| Feature Importances | Horizontal Bar | feature, importance |
| Confusion Matrix Heat (RF) | Heatmap | predicted, actual, count |

**Tableau Extract**: `tableau_exports/dash2_model_metrics.csv`, `dash2_feature_importance.csv`


In [ ]:
# =============================================================================
# DASHBOARD 2 — MODEL PERFORMANCE & FEATURE ANALYSIS
# =============================================================================

# ── Export Tableau Extracts ─────────────────────────────────────────────────
model_metrics.to_csv(f"{TABLEAU_DIR}/dash2_model_metrics.csv", index=False)
feat_imp.to_csv(f"{TABLEAU_DIR}/dash2_feature_importance.csv", index=False)
train_times.to_csv(f"{TABLEAU_DIR}/dash2_train_times.csv", index=False)

# ── Dashboard Layout ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor=TABLEAU_BG)
fig.suptitle("Dashboard 2 · Model Performance & Feature Analysis",
             fontsize=18, fontweight="bold", color="#333333")
axes = axes.flatten()

# 2-A  Accuracy & F1 grouped bar
ax = axes[0]
x = np.arange(len(model_metrics))
w = 0.35
bars1 = ax.bar(x - w/2, model_metrics["Accuracy"], w,
               label="Accuracy", color=TABLEAU_BLUE, edgecolor="white")
bars2 = ax.bar(x + w/2, model_metrics["F1"], w,
               label="F1 Score", color=TABLEAU_ORANGE, edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(model_metrics["Model"], rotation=20, ha="right", fontsize=8)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("Accuracy & F1 by Model\n(LOD: FIXED [Model] : MAX([Accuracy]))", fontsize=11)
ax.legend(fontsize=9)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7)

# 2-B  Training time log-scale comparison
ax = axes[1]
tt = train_times.sort_values("time_s", ascending=False)
color_map = {f: c for f, c in zip(tt["framework"].unique(), [TABLEAU_BLUE, TABLEAU_ORANGE, TABLEAU_GREEN])}
bar_cols = [color_map.get(f, TABLEAU_PURPLE) for f in tt["framework"]]
ax.barh(tt["label"], tt["time_s"], color=bar_cols, edgecolor="white")
ax.set_xscale("log"); ax.set_xlabel("Training Time (s, log scale)")
ax.set_title("Training Time: MLlib vs Sklearn\n(Distributed vs Single-Node)", fontsize=11)
legend_handles = [plt.Rectangle((0,0),1,1, color=color_map[k]) for k in color_map]
ax.legend(legend_handles, list(color_map.keys()), fontsize=8, loc="lower right")

# 2-C  Feature importance horizontal bar
ax = axes[2]
fi = feat_imp.head(15).sort_values("importance")
ax.barh(fi["feature"], fi["importance"], color=TABLEAU_TEAL, edgecolor="white")
ax.set_xlabel("Importance Score"); ax.set_title("Top 15 Feature Importances (Random Forest)", fontsize=11)
for i, v in enumerate(fi["importance"]):
    ax.text(v + 0.0005, i, f"{v:.4f}", va="center", fontsize=7)

# 2-D  Radar / spider chart — multi-metric model comparison
ax = axes[3]
metrics_radar = ["Accuracy", "F1"]
mlib_mask = model_metrics["Type"] == "Distributed"
sklearn_mask = model_metrics["Type"] == "Single-Node"
mlib_means = model_metrics[mlib_mask][metrics_radar].mean()
sk_means    = model_metrics[sklearn_mask][metrics_radar].mean()

categories  = ["Accuracy", "F1 Score"]
x_pos = np.arange(len(categories))
ax.bar(x_pos - 0.2, [mlib_means["Accuracy"], mlib_means["F1"]], 0.4,
       label="MLlib (Avg)", color=TABLEAU_BLUE, edgecolor="white")
ax.bar(x_pos + 0.2, [sk_means["Accuracy"], sk_means["F1"]], 0.4,
       label="Sklearn (Avg)", color=TABLEAU_ORANGE, edgecolor="white")
ax.set_xticks(x_pos); ax.set_xticklabels(categories)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("MLlib vs Sklearn\nAverage Performance", fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(f"{TABLEAU_DIR}/dashboard2_model_performance.png", dpi=150,
            bbox_inches="tight", facecolor=TABLEAU_BG)
plt.show()
print(f"  Saved: {TABLEAU_DIR}/dashboard2_model_performance.png")
print(f"  Extracts: dash2_model_metrics.csv, dash2_feature_importance.csv, dash2_train_times.csv")

## 3a — Dashboard 3: Business Insights & Operational Intelligence

| View | Chart Type | Key Fields |
|------|-----------|------------|
| Borough Resolution Heatmap | Heat Map | borough, resolution_hours |
| Top Complaint Types | Horizontal Bar | complaint_type, count |
| Agency Performance Scorecard | Bullet Chart | agency, avg_resolution, resolution_rate |
| Request Channel Distribution | Pie / Donut | channel, count |
| Hourly Demand Pattern | Area / Line | hour_of_day, complaint_count |
| Monthly Trend | Dual-Axis Line | month, count, avg_resolution_hours |

**Tableau Extract**: `tableau_exports/dash3_borough_perf.csv`, `dash3_top_complaints.csv`, `dash3_agency_perf.csv`

> **LOD Expression Example (Tableau)**:  
> `{FIXED [Borough] : AVG([Resolution Hours])}` — compute per-borough average regardless of filter context  
> `{FIXED [Complaint Type], [Year] : COUNT([UniqueKey])}` — year-over-year complaint volumes


In [ ]:
# =============================================================================
# DASHBOARD 3 — BUSINESS INSIGHTS & OPERATIONAL INTELLIGENCE
# =============================================================================

# ── Export Tableau Extracts ─────────────────────────────────────────────────
borough_perf.to_csv(f"{TABLEAU_DIR}/dash3_borough_perf.csv", index=False)
top_complaints.to_csv(f"{TABLEAU_DIR}/dash3_top_complaints.csv", index=False)
agency_perf.to_csv(f"{TABLEAU_DIR}/dash3_agency_perf.csv", index=False)
channel_dist.to_csv(f"{TABLEAU_DIR}/dash3_channel_dist.csv", index=False)
hourly_pattern.to_csv(f"{TABLEAU_DIR}/dash3_hourly_pattern.csv", index=False)
monthly_trend.to_csv(f"{TABLEAU_DIR}/dash3_monthly_trend.csv", index=False)

# ── Dashboard Layout (3×2 grid) ─────────────────────────────────────────────
fig = plt.figure(figsize=(20, 15), facecolor=TABLEAU_BG)
fig.suptitle("Dashboard 3 · Business Insights & Operational Intelligence",
             fontsize=18, fontweight="bold", color="#333333", y=0.99)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.38)

# 3-A  Borough performance heatmap
ax1 = fig.add_subplot(gs[0, :2])
if not borough_perf.empty and "avg_resolution_hours" in borough_perf.columns:
    bp_sorted = borough_perf.sort_values("avg_resolution_hours", ascending=False)
    bar_cols3 = [TABLEAU_RED if v > borough_perf["avg_resolution_hours"].quantile(0.75)
                 else TABLEAU_ORANGE if v > borough_perf["avg_resolution_hours"].median()
                 else TABLEAU_GREEN
                 for v in bp_sorted["avg_resolution_hours"]]
    ax1.barh(bp_sorted["borough"], bp_sorted["avg_resolution_hours"],
             color=bar_cols3, edgecolor="white")
    if "resolution_rate" in bp_sorted.columns:
        ax1b = ax1.twiny()
        ax1b.plot(bp_sorted["resolution_rate"] * 100, range(len(bp_sorted)),
                  "D--", color=TABLEAU_BLUE, ms=5, lw=1.5, label="Resolution Rate %")
        ax1b.set_xlabel("Resolution Rate (%)", color=TABLEAU_BLUE)
        ax1b.legend(fontsize=8, loc="lower right")
ax1.set_xlabel("Avg Resolution Hours")
ax1.set_title("Borough Performance: Resolution Time vs Rate\n"
              "(LOD: {FIXED [Borough] : AVG([Resolution Hours])})", fontsize=11)

# 3-B  Top complaint types
ax2 = fig.add_subplot(gs[0, 2])
if not top_complaints.empty:
    tc = top_complaints.head(12)
    ax2.barh(tc["complaint_type"], tc["count"],
             color=PALETTE[:len(tc)], edgecolor="white")
ax2.set_xlabel("Count"); ax2.set_title("Top Complaint Types", fontsize=11)
ax2.tick_params(axis="y", labelsize=7)

# 3-C  Agency performance bullet chart
ax3 = fig.add_subplot(gs[1, :2])
if not agency_perf.empty:
    ap = agency_perf.sort_values("complaint_count", ascending=False).head(12)
    x_a = np.arange(len(ap))
    ax3.bar(x_a, ap["complaint_count"], color=TABLEAU_BLUE, alpha=0.7,
            label="Total Complaints", edgecolor="white")
    if "avg_resolution_hours" in ap.columns:
        ax3b = ax3.twinx()
        ax3b.plot(x_a, ap["avg_resolution_hours"], "o-",
                  color=TABLEAU_RED, ms=5, lw=2, label="Avg Resolution (h)")
        ax3b.set_ylabel("Avg Resolution Hours", color=TABLEAU_RED)
        ax3b.legend(fontsize=8, loc="upper right")
    ax3.set_xticks(x_a)
    ax3.set_xticklabels(ap["agency"], rotation=30, ha="right", fontsize=8)
ax3.set_ylabel("Complaint Count"); ax3.set_title("Agency Performance Scorecard", fontsize=11)
ax3.legend(fontsize=8, loc="upper left")

# 3-D  Channel distribution donut
ax4 = fig.add_subplot(gs[1, 2])
if not channel_dist.empty:
    wedges, texts, autotexts = ax4.pie(
        channel_dist["count"], labels=channel_dist["channel"],
        autopct="%1.1f%%", colors=PALETTE[:len(channel_dist)],
        pctdistance=0.8, wedgeprops=dict(width=0.55, edgecolor="white"),
        textprops={"fontsize": 8})
    for at in autotexts: at.set_fontsize(7)
ax4.set_title("Request Channel Split\n(Open/Closed 311 vs Others)", fontsize=11)

# 3-E  Hourly demand heatmap-like area chart
ax5 = fig.add_subplot(gs[2, :2])
if not hourly_pattern.empty:
    ax5.fill_between(hourly_pattern["hour_of_day"], hourly_pattern["count"],
                     alpha=0.4, color=TABLEAU_TEAL)
    ax5.plot(hourly_pattern["hour_of_day"], hourly_pattern["count"],
             color=TABLEAU_TEAL, lw=2)
    peak_h = hourly_pattern.loc[hourly_pattern["count"].idxmax()]
    ax5.annotate(f"Peak\n{peak_h['hour_of_day']:02d}:00",
                 xy=(peak_h["hour_of_day"], peak_h["count"]),
                 xytext=(peak_h["hour_of_day"] + 2, peak_h["count"] * 0.92),
                 arrowprops=dict(arrowstyle="->", color=TABLEAU_RED), fontsize=8,
                 color=TABLEAU_RED)
    ax5.set_xticks(range(0, 24, 2))
    ax5.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)], rotation=30, fontsize=7)
ax5.set_ylabel("Complaint Volume"); ax5.set_title("Hourly Demand Pattern (24h)", fontsize=11)

# 3-F  Monthly trend dual axis
ax6 = fig.add_subplot(gs[2, 2])
if not monthly_trend.empty and len(monthly_trend) > 1:
    x_m = range(len(monthly_trend))
    ax6.bar(x_m, monthly_trend["count"], color=TABLEAU_BLUE, alpha=0.6,
            label="Monthly Complaints")
    if "avg_resolution_hours" in monthly_trend.columns:
        ax6b = ax6.twinx()
        ax6b.plot(x_m, monthly_trend["avg_resolution_hours"], "s-",
                  color=TABLEAU_ORANGE, ms=4, lw=1.5, label="Avg Res. Time (h)")
        ax6b.set_ylabel("Hours", color=TABLEAU_ORANGE)
        ax6b.legend(fontsize=7, loc="upper right")
    xticks = list(x_m)[::max(1, len(monthly_trend)//6)]
    ax6.set_xticks(xticks)
    ax6.set_xticklabels(monthly_trend["period"].iloc[xticks], rotation=30, fontsize=7)
ax6.set_ylabel("Count"); ax6.set_title("Monthly Complaint Trend", fontsize=11)
ax6.legend(fontsize=7, loc="upper left")

plt.savefig(f"{TABLEAU_DIR}/dashboard3_business_insights.png", dpi=150,
            bbox_inches="tight", facecolor=TABLEAU_BG)
plt.show()
print(f"  Saved: {TABLEAU_DIR}/dashboard3_business_insights.png")
print(f"  Extracts: dash3_borough_perf.csv, dash3_top_complaints.csv, dash3_agency_perf.csv, dash3_channel_dist.csv")

## 3a — Dashboard 4: Scalability, Cost & Infrastructure

| View | Chart Type | Key Fields |
|------|-----------|------------|
| Strong Scaling (Speedup vs Partitions) | Dual-Axis Line | partitions, speedup, efficiency_pct |
| Weak Scaling (Throughput vs Load) | Line | partitions, records_per_partition, time_s |
| Bottleneck Analysis | Sorted Bar | operation, duration_s |
| Cost-Performance Trade-off | Bubble Scatter | instance, cost_per_hr, throughput, cost_per_record |

**Tableau Extract**: `tableau_exports/dash4_scaling.csv`, `dash4_cost.csv`, `dash4_bottleneck.csv`

> **Parameter Control Example**:  
> Create a Tableau Integer Parameter `[Target Throughput (records/s)]` to dynamically highlight  
> instances in the Cost-Performance scatter that exceed the threshold.


In [ ]:
# =============================================================================
# DASHBOARD 4 — SCALABILITY, COST & INFRASTRUCTURE
# =============================================================================

# ── Export Tableau Extracts ─────────────────────────────────────────────────
strong_sc_df.to_csv(f"{TABLEAU_DIR}/dash4_strong_scaling.csv", index=False)
weak_sc_df.to_csv(f"{TABLEAU_DIR}/dash4_weak_scaling.csv", index=False)
cost_df.to_csv(f"{TABLEAU_DIR}/dash4_cost.csv", index=False)
bottleneck_df.to_csv(f"{TABLEAU_DIR}/dash4_bottleneck.csv", index=False)

# ── Dashboard Layout (2×2) ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 12), facecolor=TABLEAU_BG)
fig.suptitle("Dashboard 4 · Scalability, Cost & Infrastructure",
             fontsize=18, fontweight="bold", color="#333333")

# 4-A  Strong scaling: speedup + efficiency
ax = axes[0, 0]
if not strong_sc_df.empty and "speedup" in strong_sc_df.columns:
    ax.plot(strong_sc_df["partitions"], strong_sc_df["speedup"],
            "o-", color=TABLEAU_BLUE, lw=2, ms=7, label="Actual Speedup")
    # Ideal line calculation
    ideal_x = strong_sc_df["partitions"]
    ideal_y = strong_sc_df["partitions"] / strong_sc_df["partitions"].iloc[0]
    ax.plot(ideal_x, ideal_y, "--", color=TABLEAU_GREEN, lw=1.5, label="Ideal Linear Speedup")
    ax.fill_between(ideal_x, ideal_y, strong_sc_df["speedup"], alpha=0.12, color=TABLEAU_RED, label="Overhead Gap")

    if "efficiency_pct" in strong_sc_df.columns:
        ax2 = ax.twinx()
        ax2.plot(strong_sc_df["partitions"], strong_sc_df["efficiency_pct"],
                 "s--", color=TABLEAU_ORANGE, ms=5, lw=1.5, label="Efficiency %")
        ax2.set_ylabel("Parallel Efficiency (%)", color=TABLEAU_ORANGE)
        ax2.set_ylim(0, 110)
        ax2.legend(fontsize=8, loc="center right")
ax.set_xlabel("Number of Partitions"); ax.set_ylabel("Speedup Factor")
ax.set_title("Strong Scaling Analysis\n(Fixed Data ≈ 200K records)", fontsize=11)
ax.legend(fontsize=8, loc="upper left")

# 4-B  Weak scaling: time vs load
ax = axes[0, 1]
if not weak_sc_df.empty and "time" in weak_sc_df.columns:
    # Using 'time' column from weak_scaling_results
    ax.plot(weak_sc_df["partitions"], weak_sc_df["time"],
            "o-", color=TABLEAU_TEAL, lw=2, ms=7, label="Wall Time")
    ax.axhline(weak_sc_df["time"].iloc[0], color=TABLEAU_GREEN, ls="--",
               lw=1.5, label="Ideal (constant)")
    ax.fill_between(weak_sc_df["partitions"],
                    weak_sc_df["time"].iloc[0],
                    weak_sc_df["time"], alpha=0.15, color=TABLEAU_ORANGE)
    ax.set_xlabel("Partitions (Proportional Data Growth)")
    ax.set_ylabel("Processing Time (s)")
ax.set_title("Weak Scaling Analysis\n(Constant work per partition)", fontsize=11)
ax.legend(fontsize=8)

# 4-C  Bottleneck sorted bar
ax = axes[1, 0]
if not bottleneck_df.empty and "duration_s" in bottleneck_df.columns:
    bd_plot = bottleneck_df.sort_values("duration_s", ascending=True)
    cmap_bot = [TABLEAU_RED if v == bd_plot["duration_s"].max() else TABLEAU_TEAL for v in bd_plot["duration_s"]]
    ax.barh(bd_plot["operation"], bd_plot["duration_s"], color=cmap_bot, edgecolor="white")
    for i, v in enumerate(bd_plot["duration_s"]):
        ax.text(v + 0.01, i, f"{v:.2f}s", va="center", fontsize=8)
    ax.set_title(f"Infrastructure Bottleneck Analysis", fontsize=11)
ax.set_xlabel("Duration (seconds)")

# 4-D  Cost-Performance bubble scatter
ax = axes[1, 1]
if not cost_df.empty:
    # Use metrics calculated in dash preparation
    # X: Cost per hour, Y: Throughput (Records/sec)
    sc = ax.scatter(cost_df["cost_per_hour"], cost_df["throughput"],
                    s=cost_df["vcpu"]*50, c=range(len(cost_df)),
                    cmap="viridis", alpha=0.85, edgecolors="grey", linewidths=0.5)
    for _, row in cost_df.iterrows():
        ax.annotate(row["instance"], (row["cost_per_hour"], row["throughput"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)

    ax.set_xlabel("Instance Cost per Hour ($)")
    ax.set_ylabel("Throughput (Records/sec)")
    ax.set_title("Cost-Performance Trade-off\n(Bubble Size = vCPU count)", fontsize=11)

plt.tight_layout()
plt.savefig(f"{TABLEAU_DIR}/dashboard4_scalability_cost.png", dpi=150, bbox_inches="tight", facecolor=TABLEAU_BG)
plt.show()
print(f"  Saved: {TABLEAU_DIR}/dashboard4_scalability_cost.png")

In [ ]:
# =============================================================================
# 3c — ANNOTATIONS, EXPORT SUMMARY & FINAL VERIFICATION
# =============================================================================
import os

# ── Annotated summary figure (key insights from all 4 dashboards) ───────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=TABLEAU_BG)
fig.suptitle("311 Data — Story at a Glance (Annotated Key Insights)",
             fontsize=15, fontweight="bold", color="#333333")

# Left panel: data quality vs model performance summary
ax_left = axes[0]
ax_left.set_xlim(0, 10); ax_left.set_ylim(0, 10); ax_left.axis("off")

story_bullets = [
    ("D1", TABLEAU_BLUE,   "Data Quality",  f"Overall completeness: {dq_nulls['completeness'].mean():.1f}%"),
    ("D1", TABLEAU_ORANGE, "Pipeline",      f"{len(dq_partitions)} Spark partitions; mean size ~{dq_partitions['records'].mean():.0f} rows"),
    ("D2", TABLEAU_GREEN,  "Best Model",    f"Highest F1: {model_metrics.loc[model_metrics['F1'].idxmax(), 'Model']} ({model_metrics['F1'].max():.3f})"),
    ("D2", TABLEAU_RED,    "MLlib Speed",   "Distributed training avoids single-node memory limits"),
    ("D3", TABLEAU_TEAL,   "Top Borough",   borough_perf.loc[borough_perf['avg_resolution_hours'].idxmin(), 'borough'] if not borough_perf.empty else "N/A"),
    ("D3", TABLEAU_PURPLE, "Top Complaint", top_complaints.iloc[0]["complaint_type"] if not top_complaints.empty else "N/A"),
]

for i, (tag, color, title, desc) in enumerate(story_bullets):
    y_pos = 9 - i * 1.45
    ax_left.add_patch(plt.Rectangle((0.1, y_pos - 0.5), 0.7, 1.0, color=color, alpha=0.2))
    ax_left.text(0.5, y_pos + 0.1, f"[{tag}]", fontsize=8, color=color, fontweight="bold")
    ax_left.text(1.1, y_pos + 0.1, title, fontsize=9, fontweight="bold", color="#333")
    ax_left.text(1.1, y_pos - 0.22, desc, fontsize=8, color="#555")

# Right panel: scalability insight scatter (cost vs throughput)
ax_right = axes[1]
if not cost_df.empty and "cost_per_hour" in cost_df.columns and "throughput" in cost_df.columns:
    sc2 = ax_right.scatter(cost_df["cost_per_hour"], cost_df["throughput"],
                           s=200, c=range(len(cost_df)), cmap="viridis",
                           edgecolors="grey", linewidths=0.5, zorder=5)
    for _, row in cost_df.iterrows():
        ax_right.annotate(row["instance"], (row["cost_per_hour"], row["throughput"]),
                          textcoords="offset points", xytext=(6, 4), fontsize=8)
    # Best value annotation
    best_idx = (cost_df["throughput"] / cost_df["cost_per_hour"]).idxmax()
    best_row  = cost_df.loc[best_idx]
    ax_right.annotate("Best\nValue",
                      xy=(best_row["cost_per_hour"], best_row["throughput"]),
                      xytext=(best_row["cost_per_hour"] * 1.1, best_row["throughput"] * 0.9),
                      arrowprops=dict(arrowstyle="->", color=TABLEAU_GREEN, lw=2),
                      fontsize=9, color=TABLEAU_GREEN, fontweight="bold")
ax_right.set_xlabel("Cost / Hour ($)"); ax_right.set_ylabel("Throughput (records/s)")
ax_right.set_title("D4: Cloud Cost vs Throughput\n(Optimal Region = Top-Left)", fontsize=11)

plt.tight_layout()
plt.savefig(f"{TABLEAU_DIR}/story_at_a_glance.png", dpi=150,
            bbox_inches="tight", facecolor=TABLEAU_BG)
plt.show()
print(f"  Saved: {TABLEAU_DIR}/story_at_a_glance.png")

# ── All exports verified ────────────────────────────────────────────────────
expected_exports = [
    "dash1_data_quality.csv", "dash1_partitions.csv", "dash1_ingestion_volume.csv",
    "dash2_model_metrics.csv", "dash2_feature_importance.csv", "dash2_train_times.csv",
    "dash3_borough_perf.csv", "dash3_top_complaints.csv", "dash3_agency_perf.csv",
    "dash3_channel_dist.csv", "dash3_hourly_pattern.csv", "dash3_monthly_trend.csv",
    "dash4_strong_scaling.csv", "dash4_weak_scaling.csv", "dash4_cost.csv",
    "dash4_bottleneck.csv",
    "dashboard1_data_quality.png", "dashboard2_model_performance.png",
    "dashboard3_business_insights.png", "dashboard4_scalability_cost.png",
    "story_at_a_glance.png",
]
print("\n── Export Manifest ───────────────────────────────────────────")
for fname in expected_exports:
    fpath = os.path.join(TABLEAU_DIR, fname)
    status = "✓" if os.path.exists(fpath) else "⚠ (will exist after cell execution)"
    size_kb = f"{os.path.getsize(fpath)/1024:.1f} KB" if os.path.exists(fpath) else ""
    print(f"  {status}  {fname:50s}  {size_kb}")

print(f"\n  Total files expected in '{TABLEAU_DIR}/': {len(expected_exports)}")

## 4a-i — Temporal (Time-Aware) Train/Validation/Test Split

**Why temporal splits matter for 311 data:**
- Service request patterns exhibit seasonality and drift (e.g., heating complaints in winter)
- Traditional random splits cause **data leakage** — future records inform past predictions
- Temporal ordering ensures the model is evaluated on truly unseen future data

| Split | Date Range | Records | Purpose |
|-------|-----------|---------|---------|
| Train | Oldest 60% | ~120K | Pattern learning |
| Validation | Next 20% | ~40K | Hyperparameter tuning |
| Test | Newest 20% | ~40K | Final hold-out evaluation |


In [ ]:
# =============================================================================
# 4a-i — TEMPORAL TRAIN/VALIDATION/TEST SPLIT
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import time

# ── Load prepared features from Part 2 ──────────────────────────────────────
# Re-read from Parquet which contains processed columns
PARTITIONED_DIR = "311_data_parquet/partitioned_by_borough"
df_temporal = spark.read.parquet(PARTITIONED_DIR)

# ── Use existing parsed timestamp for temporal ordering ────────────────────
# The column 'created_date_parsed' is already a TimestampType from our ETL step
df_temporal = df_temporal.withColumn("created_ts", F.col("created_date_parsed"))
df_temporal = df_temporal.filter(F.col("created_ts").isNotNull())

# ── Compute temporal percentiles for split boundaries ───────────────────────
# Convert to unix timestamp (epoch) for quantile calculation
df_temporal = df_temporal.withColumn("ts_epoch", F.unix_timestamp("created_ts"))

epoch_quantiles = df_temporal.approxQuantile("ts_epoch", [0.6, 0.8], 0.01)

train_cutoff  = epoch_quantiles[0]
val_cutoff    = epoch_quantiles[1]

# ── Temporal splits ─────────────────────────────────────────────────────────
df_train_temp = df_temporal.filter(F.col("ts_epoch") <= train_cutoff)
df_val_temp   = df_temporal.filter((F.col("ts_epoch") > train_cutoff) & (F.col("ts_epoch") <= val_cutoff))
df_test_temp  = df_temporal.filter(F.col("ts_epoch") > val_cutoff)

# Cache for reuse
df_train_temp.cache()
df_val_temp.cache()
df_test_temp.cache()

train_count = df_train_temp.count()
val_count   = df_val_temp.count()
test_count  = df_test_temp.count()

print("=" * 60)
print("TEMPORAL SPLIT SUMMARY (time-ordered, no data leakage)")
print("=" * 60)
print(f"  Train      : {train_count:>8,} records  (oldest 60%)")
print(f"  Validation : {val_count:>8,} records  (next 20%)")
print(f"  Test       : {test_count:>8,} records  (newest 20%)")
print(f"  Total      : {train_count + val_count + test_count:>8,}")
print()

# Date ranges
train_range = df_train_temp.agg(F.min("created_ts"), F.max("created_ts")).collect()[0]
val_range   = df_val_temp.agg(F.min("created_ts"), F.max("created_ts")).collect()[0]
test_range  = df_test_temp.agg(F.min("created_ts"), F.max("created_ts")).collect()[0]

print(f"  Train period      : {train_range[0]} → {train_range[1]}")
print(f"  Validation period : {val_range[0]} → {val_range[1]}")
print(f"  Test period       : {test_range[0]} → {test_range[1]}")

## 4a-ii — Cross-Validation with Stratification (Imbalanced Data)

**Challenge**: 311 resolution categories are often imbalanced (e.g., most complaints resolved quickly).

**Solution**:
1. **Weighted sampling** to ensure each fold contains proportional class representation
2. **Stratified K-Fold** via custom fold assignment based on label distribution
3. **Class weights** in the classifier to penalise minority class errors

| Fold | Class 0 | Class 1 | Class 2 | Stratified? |
|------|---------|---------|---------|-------------|
| 1 | ~33% | ~33% | ~34% | |
| 2 | ~33% | ~33% | ~34% | |
| 3 | ~33% | ~33% | ~34% | |


In [ ]:
# =============================================================================
# 4a-ii — STRATIFIED CROSS-VALIDATION FOR IMBALANCED DATA
# =============================================================================
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql.functions import rand, row_number, floor, lit
from pyspark.sql.window import Window
import numpy as np

NUM_FOLDS = 3  # Keep small for fast execution

# ── Create resolution_category label (fast < 24h, medium 24-72h, slow > 72h) ─
# FIXED: Using closed_date_parsed and created_date_parsed from processed Parquet
df_stratified = df_train_temp.withColumn(
    "label",
    F.when(F.col("resolution_hours") < 24, 0)
     .when(F.col("resolution_hours") < 72, 1)
     .otherwise(2)
).filter(F.col("label").isNotNull() & (F.col("resolution_hours") >= 0))

# ── Class distribution ──
class_counts = df_stratified.groupBy("label").count().orderBy("label").collect()
total_rows   = df_stratified.count()

print("=" * 60)
print("CLASS DISTRIBUTION (before stratified CV)")
print("=" * 60)
for row in class_counts:
    pct = row["count"] / total_rows * 100
    print(f"  Class {row['label']}: {row['count']:>8,} ({pct:>5.1f}%)")
print(f"  Total : {total_rows:>8,}")

# ── Assign stratified fold indices ──
window_spec = Window.partitionBy("label").orderBy(rand(seed=42))
df_stratified = df_stratified.withColumn("row_in_class", row_number().over(window_spec))
df_stratified = df_stratified.withColumn(
    "fold_id",
    (F.col("row_in_class") - 1) % NUM_FOLDS
)

# ── Verify stratification ──
print("\n── Fold Distribution by Class ──")
fold_dist = df_stratified.groupBy("fold_id", "label").count().orderBy("fold_id", "label").collect()
fold_summary = {}
for row in fold_dist:
    fid, lbl, cnt = row["fold_id"], row["label"], row["count"]
    if fid not in fold_summary:
        fold_summary[fid] = {}
    fold_summary[fid][lbl] = cnt

for fid in sorted(fold_summary.keys()):
    fold_total = sum(fold_summary[fid].values())
    class_pcts = " | ".join([f"C{c}: {fold_summary[fid].get(c, 0)/fold_total*100:.1f}%"
                             for c in sorted(fold_summary[fid].keys())])
    print(f"  Fold {fid}: {fold_total:>6,} records  ({class_pcts})")

# ── Simple feature pipeline (minimal for speed) ──
df_stratified = df_stratified.na.fill({"borough_clean": "UNKNOWN", "agency_clean": "OTHER"})

indexer_borough = StringIndexer(inputCol="borough_clean", outputCol="borough_idx", handleInvalid="keep")
indexer_agency  = StringIndexer(inputCol="agency_clean", outputCol="agency_idx", handleInvalid="keep")
assembler = VectorAssembler(inputCols=["borough_idx", "agency_idx"], outputCol="features")
mini_pipeline = Pipeline(stages=[indexer_borough, indexer_agency, assembler])
mini_model = mini_pipeline.fit(df_stratified)
df_ready = mini_model.transform(df_stratified).select("features", "label", "fold_id")
df_ready.cache()

# ── Run stratified K-fold cross-validation ──
evaluator = MulticlassClassificationEvaluator(predictionCol="prediction", labelCol="label", metricName="f1")
stratified_cv_results = []

t_cv_start = time.time()
for fold in range(NUM_FOLDS):
    df_cv_train = df_ready.filter(F.col("fold_id") != fold)
    df_cv_val   = df_ready.filter(F.col("fold_id") == fold)

    rf_cv = RandomForestClassifier(
        featuresCol="features", labelCol="label",
        numTrees=10, maxDepth=5, seed=42
    )
    model_cv = rf_cv.fit(df_cv_train)
    preds = model_cv.transform(df_cv_val)

    f1 = evaluator.evaluate(preds)
    stratified_cv_results.append({"fold": fold, "f1_score": f1})
    print(f"  Fold {fold}: F1 = {f1:.4f}")

t_cv_end = time.time()

cv_f1_mean = np.mean([r["f1_score"] for r in stratified_cv_results])
cv_f1_std  = np.std([r["f1_score"] for r in stratified_cv_results])

print("\n── Stratified CV Summary ──")
print(f"  Mean F1: {cv_f1_mean:.4f} ± {cv_f1_std:.4f}")
print(f"  Total CV time: {t_cv_end - t_cv_start:.1f}s ({NUM_FOLDS} folds)")

## 4a-iii — Bootstrap Confidence Intervals (Statistical Significance)

**Purpose**: Point estimates (e.g., "F1 = 0.72") hide uncertainty. Bootstrap resampling quantifies variance.

**Method**:
1. Train a model on the full training set once
2. Draw B bootstrap samples (sampling with replacement) from **test set predictions**
3. Compute metric on each sample → distribution of B estimates
4. Report 95% CI: [2.5th percentile, 97.5th percentile]

**Interpretation**:
- If two model CIs overlap substantially → no statistically significant difference
- Narrow CI → stable estimate; wide CI → high variance (collect more data or simplify model)


In [ ]:
# =============================================================================
# 4a-iii — BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
import numpy as np

NUM_BOOTSTRAP = 100  # Reduced for speed; production would use 1000+

# ── Prepare test predictions (using trained model from CV) ─────────────────
# Re-use the last fold's model or train fresh on full train set
df_test_features = mini_model.transform(df_test_temp.na.fill({
    "borough_clean": "UNKNOWN", "agency_clean": "OTHER"
})).withColumn(
    "label",
    F.when(F.col("resolution_hours") < 24, 0)
     .when(F.col("resolution_hours") < 72, 1)
     .otherwise(2)
).filter(F.col("label").isNotNull() & (F.col("resolution_hours") >= 0)).select("features", "label")

# Train on full stratified train set
rf_bootstrap = RandomForestClassifier(
    featuresCol="features", labelCol="label",
    numTrees=15, maxDepth=5, seed=42
)
model_bootstrap = rf_bootstrap.fit(df_ready.select("features", "label"))
test_preds = model_bootstrap.transform(df_test_features)

# ── Collect predictions to driver for fast bootstrap ───────────────────────
# Sample down for speed (max 10K records)
test_sample = test_preds.select("label", "prediction").limit(10000).toPandas()
y_true = test_sample["label"].values.astype(int)
y_pred = test_sample["prediction"].values.astype(int)

n = len(y_true)
print(f"Bootstrap sample size: {n:,} records")

# ── Bootstrap resampling ───────────────────────────────────────────────────
np.random.seed(42)
bootstrap_f1       = []
bootstrap_accuracy = []
bootstrap_precision = []
bootstrap_recall   = []

t_boot_start = time.time()
for b in range(NUM_BOOTSTRAP):
    indices = np.random.choice(n, size=n, replace=True)
    y_true_b = y_true[indices]
    y_pred_b = y_pred[indices]

    bootstrap_f1.append(f1_score(y_true_b, y_pred_b, average="weighted", zero_division=0))
    bootstrap_accuracy.append(accuracy_score(y_true_b, y_pred_b))
    bootstrap_precision.append(precision_score(y_true_b, y_pred_b, average="weighted", zero_division=0))
    bootstrap_recall.append(recall_score(y_true_b, y_pred_b, average="weighted", zero_division=0))

t_boot_end = time.time()

# ── Compute confidence intervals ───────────────────────────────────────────
def ci_95(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

ci_results = {
    "F1 (weighted)": (np.mean(bootstrap_f1), *ci_95(bootstrap_f1)),
    "Accuracy":      (np.mean(bootstrap_accuracy), *ci_95(bootstrap_accuracy)),
    "Precision":     (np.mean(bootstrap_precision), *ci_95(bootstrap_precision)),
    "Recall":        (np.mean(bootstrap_recall), *ci_95(bootstrap_recall)),
}

print("\n" + "=" * 60)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("=" * 60)
print(f"  Bootstrap iterations: {NUM_BOOTSTRAP}")
print(f"  Computation time: {t_boot_end - t_boot_start:.2f}s")
print("-" * 60)
for metric, (mean_val, lo, hi) in ci_results.items():
    width = hi - lo
    print(f"  {metric:<16} {mean_val:.4f}  [{lo:.4f}, {hi:.4f}]  (width={width:.4f})")

# ── Store for visualization ────────────────────────────────────────────────
bootstrap_results = {
    "f1": bootstrap_f1,
    "accuracy": bootstrap_accuracy,
    "precision": bootstrap_precision,
    "recall": bootstrap_recall,
    "ci_results": ci_results
}

## 4a-iv — Business Metric Alignment (Expected Profit / CLV Proxy)

**Why business metrics?**  
ML metrics (F1, AUC) don't translate directly to organisational value.  
Decision-makers care about:
- **Expected operational cost** (wrong predictions → wasted resources)
- **Service Level Agreement (SLA) adherence** → fines / reputation cost
- **Customer Lifetime Value proxy** → resident satisfaction, future service usage

**Cost Matrix for 311 Resolution Prediction**:
| Predicted \ Actual | Fast (0) | Medium (1) | Slow (2) |
|--------------------|----------|------------|----------|
| Fast (0)           | $0       | -$50       | -$200    |
| Medium (1)         | -$20     | $0         | -$100    |
| Slow (2)           | -$30     | -$40       | $0       |

- "Predicting Fast but actually Slow" → -$200 (SLA breach, emergency escalation)
- "Predicting Slow but actually Fast" → -$30 (over-allocation of resources)


In [ ]:
# =============================================================================
# 4a-iv — BUSINESS METRIC ALIGNMENT (Cost Matrix & CLV Proxy)
# =============================================================================
from sklearn.metrics import confusion_matrix
import pandas as pd

# ── Define cost matrix ─────────────────────────────────────────────────────
# Rows = predicted, Columns = actual
# Cost[pred][actual] = business impact in $
COST_MATRIX = np.array([
    [   0,  -50, -200],   # Predicted Fast
    [ -20,    0, -100],   # Predicted Medium
    [ -30,  -40,    0],   # Predicted Slow
])

# Labels for readability
CLASS_NAMES = ["Fast (<24h)", "Medium (24-72h)", "Slow (>72h)"]

# ── Compute confusion matrix ───────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

print("=" * 60)
print("CONFUSION MATRIX (Test Set)")
print("=" * 60)
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = "Actual →"
cm_df.columns.name = "↓ Predicted"
print(cm_df)
print()

# ── Compute expected cost per prediction ───────────────────────────────────
total_predictions = cm.sum()
# Element-wise multiply counts by costs, then sum
total_cost = np.sum(cm * COST_MATRIX.T)  # Transpose to align pred-actual
avg_cost_per_prediction = total_cost / total_predictions

print("=" * 60)
print("BUSINESS COST ANALYSIS")
print("=" * 60)
print(f"  Total predictions : {total_predictions:,}")
print(f"  Total expected cost: ${total_cost:,.2f}")
print(f"  Avg cost/prediction: ${avg_cost_per_prediction:,.2f}")
print()

# ── Breakdown by error type ────────────────────────────────────────────────
print("── Cost Breakdown by Error Type ──")
error_costs = []
for pred in range(3):
    for actual in range(3):
        if pred != actual:  # Only errors
            count = cm[actual, pred]  # Note: confusion_matrix[actual, pred]
            unit_cost = COST_MATRIX[pred, actual]
            total_err_cost = count * unit_cost
            if count > 0:
                error_costs.append({
                    "Predicted": CLASS_NAMES[pred],
                    "Actual": CLASS_NAMES[actual],
                    "Count": count,
                    "Unit Cost": unit_cost,
                    "Total Cost": total_err_cost
                })

error_df = pd.DataFrame(error_costs).sort_values("Total Cost")
print(error_df.to_string(index=False))

# ── Customer Lifetime Value (CLV) Proxy ────────────────────────────────────
# CLV proxy = satisfaction score based on prediction accuracy and expectation setting
# If we correctly predict resolution time → higher resident satisfaction
def clv_proxy(y_true, y_pred):
    """
    CLV proxy: +10 for correct prediction (good expectation setting)
               -5 for predicting faster than actual (disappointed)
               -2 for predicting slower than actual (pleasant surprise, but over-alloc)
    """
    scores = []
    for true, pred in zip(y_true, y_pred):
        if pred == true:
            scores.append(10)
        elif pred < true:  # Predicted faster, actually slower (disappointment)
            scores.append(-5)
        else:  # Predicted slower, actually faster (positive surprise)
            scores.append(-2)
    return np.mean(scores)

clv_score = clv_proxy(y_true, y_pred)

print("\n── CLV Proxy (Resident Satisfaction Score) ──")
print(f"  Mean CLV Score: {clv_score:.2f} / 10.00 max")
print(f"  Interpretation: {'Good' if clv_score > 5 else 'Moderate' if clv_score > 0 else 'Poor'} expectation setting")

# ── Store business metrics ─────────────────────────────────────────────────
business_metrics = {
    "total_predictions": total_predictions,
    "total_cost": total_cost,
    "avg_cost_per_prediction": avg_cost_per_prediction,
    "clv_proxy_score": clv_score,
    "confusion_matrix": cm,
    "error_breakdown": error_df
}

In [ ]:
# =============================================================================
# PART 4 — EVALUATION VISUALISATION
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

TABLEAU_BLUE='#4E79A7'; TABLEAU_ORANGE='#F28E2B'; TABLEAU_RED='#E15759'
TABLEAU_TEAL='#76B7B2'; TABLEAU_GREEN='#59A14F'; TABLEAU_BG='#F5F5F5'

fig = plt.figure(figsize=(18, 10), facecolor=TABLEAU_BG)
fig.suptitle("Part 4 — Model Evaluation Summary", fontsize=16, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 4-A  Bootstrap CI distributions (histograms)
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(bootstrap_f1, bins=20, color=TABLEAU_BLUE, edgecolor="white", alpha=0.8)
ax1.axvline(ci_results["F1 (weighted)"][1], color=TABLEAU_RED, ls="--", lw=1.5, label="2.5%")
ax1.axvline(ci_results["F1 (weighted)"][2], color=TABLEAU_RED, ls="--", lw=1.5, label="97.5%")
ax1.axvline(ci_results["F1 (weighted)"][0], color=TABLEAU_GREEN, ls="-", lw=2, label="Mean")
ax1.set_title("Bootstrap F1 Distribution\n(95% CI)", fontsize=11)
ax1.set_xlabel("F1 Score"); ax1.legend(fontsize=8)

# 4-B  Stratified CV fold comparison
ax2 = fig.add_subplot(gs[0, 1])
fold_ids = [r["fold"] for r in stratified_cv_results]
fold_f1s = [r["f1_score"] for r in stratified_cv_results]
bars = ax2.bar(fold_ids, fold_f1s, color=TABLEAU_TEAL, edgecolor="white")
ax2.axhline(cv_f1_mean, color=TABLEAU_RED, ls="--", lw=1.5, label=f"Mean={cv_f1_mean:.3f}")
ax2.set_ylim(0, 1.05)
for bar, val in zip(bars, fold_f1s):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{val:.3f}",
             ha="center", fontsize=9)
ax2.set_xticks(fold_ids); ax2.set_xlabel("Fold"); ax2.set_ylabel("F1 Score")
ax2.set_title(f"Stratified {NUM_FOLDS}-Fold CV\n(F1 ± {cv_f1_std:.3f})", fontsize=11)
ax2.legend(fontsize=8)

# 4-C  Confusion matrix heatmap with cost overlay
ax3 = fig.add_subplot(gs[0, 2])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax3,
            xticklabels=[f"P:{c[:4]}" for c in CLASS_NAMES],
            yticklabels=[f"A:{c[:4]}" for c in CLASS_NAMES],
            cbar_kws={"shrink": 0.8})
ax3.set_xlabel("Predicted"); ax3.set_ylabel("Actual")
ax3.set_title("Confusion Matrix\n(Test Set)", fontsize=11)

# 4-D  Cost breakdown bar chart
ax4 = fig.add_subplot(gs[1, 0])
if len(error_df) > 0:
    err_labels = [f"{r['Predicted'][:3]}→{r['Actual'][:3]}" for _, r in error_df.iterrows()]
    err_costs = error_df["Total Cost"].values
    colors_err = [TABLEAU_RED if c < -5000 else TABLEAU_ORANGE for c in err_costs]
    ax4.barh(err_labels, err_costs, color=colors_err, edgecolor="white")
    for i, v in enumerate(err_costs):
        ax4.text(v - 50, i, f"${v:,.0f}", va="center", ha="right", fontsize=8, color="white")
ax4.set_xlabel("Total Cost ($)"); ax4.set_title("Error Cost Breakdown", fontsize=11)

# 4-E  CLV Gauge
ax5 = fig.add_subplot(gs[1, 1])
theta = np.linspace(0, np.pi, 100)
clv_norm = max(0, min(10, clv_score)) / 10
ax5.fill_between(np.cos(theta), np.sin(theta), alpha=0.1, color=TABLEAU_BLUE)
needle_ang = np.pi * (1 - clv_norm)
ax5.annotate("", xy=(np.cos(needle_ang)*0.8, np.sin(needle_ang)*0.8),
             xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=TABLEAU_GREEN, lw=3))
ax5.text(0, -0.2, f"{clv_score:.1f}/10", ha="center", fontsize=18, fontweight="bold",
         color=TABLEAU_GREEN if clv_score > 5 else TABLEAU_ORANGE)
ax5.text(0, -0.45, "CLV Proxy Score", ha="center", fontsize=10, color="#555")
ax5.set_xlim(-1.2, 1.2); ax5.set_ylim(-0.6, 1.2); ax5.set_aspect("equal"); ax5.axis("off")
ax5.set_title("Resident Satisfaction", fontsize=11)